# 末日大逃殺

城市被殭屍佔領了，唯一能逃的是撤離點那台壞掉的直升機。
全班一起把它修好、加滿油，起飛時站在旁邊就逃出去了。

你不會直接操控角色，而是寫一個策略：看地圖，決定要去哪裡、做什麼，角色就照著走。

怎麼用這份筆記本：

1. 執行「1. 設定」和「2. 安裝」，只要一次
2. 改「3. 我的策略」，再用「4. 本地測試」看看成績，不用連伺服器，可以一直試
3. 比賽時執行「5. 上場」，去大螢幕上找你的角色

## 1. 設定

把名字改成你自己的。伺服器網址講師會給。

In [ ]:
#@title 1. 設定
SERVER_URL = "https://請講師填入伺服器網址"   #@param {type:"string"}
MY_NAME  = "改成你的名字"   #@param {type:"string"}

## 2. 安裝

按執行就好，不用看內容。

In [ ]:
#@title 2. 安裝（按執行就好）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ortools"], check=False)
subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1", shell=True)
FILES = {'apocalypse_sdk.py': 'from __future__ import annotations\n\nimport heapq\nimport time\nimport types\nfrom typing import Callable, Iterable\n\nimport requests\n\n__all__ = [\n    "Game", "State", "CityGraph", "ApiError", "benchmark", "test_local", "play", "draw", "replay",\n    "move", "goto", "pickup", "dropoff", "take_medkit", "use_medkit", "heal", "revive", "wait", "fix",\n    "take_weapon", "drop", "drop_medkit", "drop_weapon",\n]\n\ndef _step(action, node=-1, task=-1, target=-1, seconds=0.0):\n    return {"action": action, "node": int(node), "task": int(task), "target": int(target), "seconds": float(seconds)}\n\ndef move(node):        return _step("move", node=node)\ndef goto(node):        return _step("goto", node=node)\ndef pickup(task):      return _step("pickup", task=task)\ndef dropoff(task):     return _step("dropoff", task=task)\ndef take_medkit():     return _step("take_medkit")\ndef use_medkit():      return _step("use_medkit")\ndef heal():            return _step("heal")\ndef revive(player_id): return _step("revive", target=player_id)\ndef wait(seconds=1.0): return _step("wait", seconds=seconds)\ndef fix(incident, seconds=10.0): return _step("fix", task=incident, seconds=seconds)\ndef take_weapon():     return _step("take_weapon")\ndef drop(task):        return _step("drop", task=task)\ndef drop_medkit():     return _step("drop_medkit")\ndef drop_weapon():     return _step("drop_weapon")\n\nclass ApiError(Exception):\n    pass\n\nclass State:\n\n    def __init__(self, raw: dict):\n        self.raw = raw\n        self.status: str = raw["status"]\n        self.time: float = raw["time"]\n        self.time_left: float = raw["time_left"]\n        self.phase: int = raw["phase"]\n        self.city_infection: float = raw["city_infection"]\n        self.rules: dict = raw["rules"]\n\n        self.you: dict = raw["you"]\n\n        self.nodes: dict[int, dict] = {n["id"]: n for n in raw["nodes"]}\n        self.edges: list[dict] = raw["edges"]\n        self.tasks: dict[int, dict] = {t["id"]: t for t in raw["tasks"]}\n\n        self.batches: dict[int, dict] = {b["id"]: b for b in raw.get("batches", [])}\n        self.players: list[dict] = raw["players"]\n\n        self.zombies: list[dict] = raw["zombies"]\n        self.zombie_speed: float = raw.get("zombie_speed", 1.3)\n\n        self.noises: list[dict] = raw.get("noises", [])\n        self.events: list[dict] = raw["events"]\n\n        self.incidents: list[dict] = raw.get("incidents", [])\n\n    @property\n    def here(self) -> int:\n        y = self.you\n        return y["next_node"] if y["next_node"] >= 0 else y["node"]\n\n    @property\n    def is_idle(self) -> bool:\n        y = self.you\n        return y["plan_index"] >= len(y["plan"]) and y["busy"] == ""\n\n    def open_tasks(self) -> list[dict]:\n        return [t for t in self.tasks.values() if t["status"] == "open" and self.can_take(t)]\n\n    def announced_tasks(self) -> list[dict]:\n        return [t for t in self.tasks.values() if t["status"] == "announced" and self.can_take(t)]\n\n    def can_take(self, t: dict) -> bool:\n        b = self.tasks_batch(t)\n        return b is None or b["owner"] in (-1, self.you.get("id", -1))\n\n    @property\n    def escape(self) -> dict:\n        return self.raw.get("escape") or {"enabled": False}\n\n    @property\n    def extraction(self) -> int:\n        return self.escape.get("extraction", -1)\n\n    def in_zone(self, node: int) -> bool:\n        e = self.escape\n        if not e.get("enabled"):\n            return True\n        n = self.nodes[node]\n        return (n["x"] - e["zone_x"]) ** 2 + (n["y"] - e["zone_y"]) ** 2 <= e["zone_radius"] ** 2\n\n    def active_incidents(self) -> list[dict]:\n        return [i for i in self.incidents if i["status"] == "active"]\n\n    def helpers(self, incident_id: int) -> int:\n        me = self.you.get("id", -1)\n        return sum(1 for p in self.players if p["id"] != me and incident_id in p.get("fixing", []))\n\n    def claimed_tasks(self) -> set[int]:\n        me = self.you.get("id", -1)\n        return {i for p in self.players if p["id"] != me for i in p.get("targets", [])}\n\n    def tasks_batch(self, t: dict):\n        return self.batches.get(t.get("batch", -1))\n\n    def my_cargo(self) -> list[dict]:\n        return [self.tasks[i] for i in self.you["cargo"] if i in self.tasks]\n\n    def nodes_of(self, type_: str) -> list[int]:\n        return [n["id"] for n in self.nodes.values() if n["type"] == type_]\n\n    def powered(self, node_id: int) -> bool:\n        return bool(self.nodes[node_id].get("powered", False))\n\n    def __repr__(self):\n        y = self.you\n        return (f"<State {self.status} t={self.time:.0f}s phase={self.phase} city={self.city_infection:.0f}% "\n                f"you@{y[\'node\']} hp={y[\'hp\']:.0f} inf={y[\'infection\']:.0f} score={y[\'score\']} "\n                f"tasks={len(self.open_tasks())} zombies={len(self.zombies)}>")\n\nclass CityGraph:\n\n    def __init__(self, state: State, danger_weight: float = 0.0, zombie_penalty: float = 0.0, zombie_radius: float = 3.0,\n                 zombie_lookahead: float = 0.0):\n        self.state = state\n        self.adj: dict[int, list[tuple[int, float]]] = {n: [] for n in state.nodes}\n        near = {n: 0.0 for n in state.nodes}\n        if zombie_penalty > 0:\n            for z in state.zombies:\n                for nid, n in state.nodes.items():\n                    if (n["x"] - z["x"]) ** 2 + (n["y"] - z["y"]) ** 2 <= zombie_radius ** 2:\n                        near[nid] += 1\n                if zombie_lookahead > 0:\n                    t, px, py = 0.0, z["x"], z["y"]\n                    for nid in z.get("path", []):\n                        n = state.nodes[nid]\n                        t += ((n["x"] - px) ** 2 + (n["y"] - py) ** 2) ** 0.5 / max(0.1, state.zombie_speed)\n                        if t > zombie_lookahead:\n                            break\n                        near[nid] += 1 - t / zombie_lookahead\n                        px, py = n["x"], n["y"]\n        for e in state.edges:\n            if e["blocked"]:\n                continue\n            a, b = e["a"], e["b"]\n            inf = (state.nodes[a]["infection"] + state.nodes[b]["infection"]) / 2\n            w = e["length"] * (1 + danger_weight * inf) + zombie_penalty * (near[a] + near[b])\n            self.adj[a].append((b, w))\n            self.adj[b].append((a, w))\n        self._cache: dict[int, tuple[dict, dict]] = {}\n\n    def dijkstra(self, src: int):\n        if src not in self._cache:\n            dist, prev = {src: 0.0}, {}\n            pq = [(0.0, src)]\n            while pq:\n                d, u = heapq.heappop(pq)\n                if d > dist.get(u, float("inf")):\n                    continue\n                for v, w in self.adj[u]:\n                    nd = d + w\n                    if nd < dist.get(v, float("inf")):\n                        dist[v], prev[v] = nd, u\n                        heapq.heappush(pq, (nd, v))\n            self._cache[src] = (dist, prev)\n        return self._cache[src]\n\n    def dist(self, a: int, b: int) -> float:\n        return self.dijkstra(a)[0].get(b, float("inf"))\n\n    def path(self, a: int, b: int) -> list[int]:\n        if a == b:\n            return []\n        dist, prev = self.dijkstra(a)\n        if b not in dist:\n            return []\n        out = [b]\n        while out[-1] != a:\n            out.append(prev[out[-1]])\n        return out[-2::-1]\n\n    def path_steps(self, a: int, b: int) -> list[dict]:\n        return [move(n) for n in self.path(a, b)]\n\n    def matrix(self, nodes: Iterable[int]) -> list[list[float]]:\n        nodes = list(nodes)\n        return [[self.dist(a, b) for b in nodes] for a in nodes]\n\n    def nearest(self, src: int, candidates: Iterable[int]):\n        best = min(candidates, key=lambda n: self.dist(src, n), default=None)\n        return best\n\nclass Game:\n    def __init__(self, url: str, token: str | None = None, timeout: float = 10.0):\n        self.url = url.rstrip("/")\n        self.token = token\n        self.timeout = timeout\n        self.session = requests.Session()\n\n        self.session.headers.update({"ngrok-skip-browser-warning": "1"})\n\n    def _req(self, method: str, path: str, **kw):\n        r = self.session.request(method, self.url + path, timeout=self.timeout, **kw)\n        try:\n            data = r.json()\n        except ValueError:\n            raise ApiError(f"{r.status_code}: {r.text[:200]}")\n        if r.status_code >= 400 or data.get("ok") is False:\n            raise ApiError(data.get("error") or f"HTTP {r.status_code}")\n        return data\n\n    def join(self, name: str, retries: int = 30, quiet: bool = False, role: str = "") -> str:\n        for i in range(retries):\n            try:\n                data = self._req("POST", "/join", json={"name": name, "role": role})\n                break\n            except requests.RequestException:\n                if i == retries - 1:\n                    raise\n                print("連不上伺服器，2 秒後重試…")\n                time.sleep(2)\n        self.token = data["token"]\n        if not quiet:\n            print(f"已加入：{name}（player_id={data[\'player_id\']}）\\ntoken = {self.token}  ← 斷線重連時用 Game(url, token=...)")\n        return self.token\n\n    def state(self) -> State:\n        return State(self._req("GET", "/state", params={"token": self.token or ""}))\n\n    def submit(self, steps: list[dict], append: bool = False, start_node: int = -1):\n        return self._req("POST", "/plan", json={"token": self.token, "steps": steps, "append": append,\n                                                "start_node": int(start_node)})\n\n    def admin_start(self, key: str = "admin"):\n        return self._req("POST", "/admin/start", json={"key": key})\n\n    def admin_reset(self, key: str = "admin"):\n        return self._req("POST", "/admin/reset", json={"key": key})\n\n    def results(self) -> dict:\n        return self._req("GET", "/results")\n\n    def wait_until_running(self, poll: float = 1.0, after_match: int | None = None) -> State:\n        printed = False\n        while True:\n            st = self.state()\n            if st.status == "running" and (after_match is None or st.raw.get("match_id") != after_match):\n                return st\n            if not printed:\n                print("等待比賽開始…")\n                printed = True\n            time.sleep(poll)\n\n    def run(self, policy: Callable[[State], list[dict] | None], interval: float = 1.0,\n            verbose: bool = True):\n        self.wait_until_running()\n        last_print = 0.0\n        while True:\n            try:\n                st = self.state()\n            except (requests.RequestException, ApiError) as e:\n                print("連線錯誤，重試中：", e)\n                time.sleep(2)\n                continue\n            if st.status == "ended":\n                print(f"比賽結束：{st.raw[\'end_reason\']}\u3000你的分數：{st.you[\'score\']}")\n                return st\n            if st.you["state"] == "eliminated":\n                print("你已被淘汰（分數保留），等待比賽結束…")\n                time.sleep(5)\n                continue\n            if verbose and st.you["last_error"]:\n                print("⚠", st.you["last_error"])\n            steps = policy(st)\n            if steps is not None and st.you["state"] == "active":\n                try:\n                    self.submit(steps, start_node=st.here)\n                except ApiError as e:\n                    if verbose:\n                        print("送出失敗：", e)\n            if verbose and time.time() - last_print > 10:\n                print(st)\n                last_print = time.time()\n            time.sleep(interval)\n\n\ndef test_local(make_policy: Callable[[], Callable], seeds=(1, 2, 3), name: str = "我", role: str = "",\n               classmates: int = 11, show: bool = True) -> list:\n    from apocalypse_local import LocalGame\n    games = []\n    for seed in seeds:\n        g = LocalGame(seed=seed, name=name, role=role, classmates=classmates)\n        try:\n            g.run(make_policy())\n        except Exception:\n            import traceback\n            print(f"種子 {seed}：你的程式在第 {g.sim.time:.0f} 秒出錯 ↓")\n            traceback.print_exc()\n            return games\n        r = g.result()\n        games.append(g)\n        if g.sim.cfg["escape"]:\n            t = int(g.sim.cfg["matchDuration"] - r["ended_at"])\n            city = (f"✅ 直升機在比賽還剩 {t // 60}:{t % 60:02d} 時起飛，{r[\'escaped_count\']}/{r[\'players\']} 人逃出" if r["class_won"]\n                    else f"❌ 直升機沒修好（零件 {r[\'parts_total\']}/{r[\'parts_required\']}）")\n            mine = f"運零件 {r[\'parts\']}、救援 {r[\'rescued\']}\u3000倒地 {r[\'downs\']} 次\u3000{\'搭上直升機\' if r[\'escaped\'] else \'沒搭上\'}"\n        else:\n            city = "✅ 全班撐過了" if r["class_won"] else f"❌ 城市在 {int(r[\'ended_at\']) // 60}:{int(r[\'ended_at\']) % 60:02d} 淪陷"\n            mine = (f"壓低感染 {r[\'city_reduced\']:.1f}%\u3000配送 {r[\'delivered\']}、救援 {r[\'rescued\']}\u3000倒地 {r[\'downs\']} 次"\n                    f"\u3000{\'撤離成功\' if r[\'alive\'] else \'沒有撤離\'}")\n        print(f"種子 {seed}：{city}\u3000｜\u3000你 {r[\'score\']:4d} 分（第 {r[\'rank\']}/{r[\'players\']} 名，同學平均 {r[\'class_avg\']:.0f}）\u3000{mine}")\n        if g.errors:\n            print(f"\u3000\u3000⚠ 有 {len(g.errors)} 個步驟被略過，例如：{g.errors[0]}")\n    if len(games) > 1:\n        rs = [g.result() for g in games]\n        print(f"全班成功 {sum(r[\'class_won\'] for r in rs)}/{len(rs)} 場；你平均 {sum(r[\'score\'] for r in rs) / len(rs):.0f} 分")\n    if show and games:\n        replay(games[-1])\n    return games\n\ndef play(make_policy: Callable[[], Callable], url: str, name: str, role: str = ""):\n    import json, os\n    game, store = Game(url), "apocalypse_token.json"\n    tokens = {}\n    if os.path.exists(store):\n        try:\n            tokens = json.load(open(store, encoding="utf-8"))\n        except ValueError:\n            pass\n    key = f"{url.rstrip(\'/\')}|{name}"\n    game.token = tokens.get(key)\n    try:\n        if game.token and game.state().you.get("id", -1) < 0:\n            game.token = None\n    except (requests.RequestException, ApiError):\n        pass\n    while game.token is None:\n        try:\n            game.join(name, quiet=True, role=role)\n            tokens[key] = game.token\n            json.dump(tokens, open(store, "w", encoding="utf-8"))\n        except ApiError as e:\n            print("還不能加入：", e, "（5 秒後再試）")\n            time.sleep(5)\n    print(f"✅ {name} 已連上，大螢幕上找找你的角色！（按 ■ 停止；改完程式重新執行會接回同一個角色）")\n    last = None\n    try:\n        while True:\n            try:\n                st = game.wait_until_running(after_match=last)\n            except (requests.RequestException, ApiError) as e:\n                print("連線中斷，重試中…", e)\n                time.sleep(3)\n                continue\n            last = st.raw.get("match_id")\n            print(f"▶ 比賽開始（種子 {st.raw.get(\'seed\')}）")\n            final = game.run(make_policy(), verbose=True)\n            won = ("✅ 成功！" if final.raw.get("class_won") else "❌ 失敗") if final.raw.get("coop") else ""\n            print(f"■ 本場結束：{won} 你 {final.you[\'score\']} 分，等待下一場…")\n    except KeyboardInterrupt:\n        print("已停止送出計畫（角色會把目前的計畫走完）。改好程式後重新執行這一格即可。")\n\n_TYPE_STYLE = {"shelter": ("#4fc3f7", "s", "避難所"), "hospital": ("#f06292", "P", "醫院"),\n               "supply": ("#ffd54f", "D", "補給站"), "factory": ("#a1887f", "s", "工廠"), "generator": ("#26c6da", "p", "發電機")}\n\ndef draw(st: State, ax=None, title: str | None = None):\n    import matplotlib.pyplot as plt\n    show = ax is None\n    if ax is None:\n        _, ax = plt.subplots(figsize=(10, 7))\n    ax.clear()\n    for e in st.edges:\n        a, b = st.nodes[e["a"]], st.nodes[e["b"]]\n        ax.plot([a["x"], b["x"]], [a["y"], b["y"]], color="red" if e["blocked"] else ("#8d6e63" if e["bridge"] else "#d0d0d0"),\n                lw=3 if e["bridge"] else 2, ls=":" if e["blocked"] else "-", zorder=1)\n    for n in st.nodes.values():\n        if n["infection"] > 0.05:\n            ax.scatter(n["x"], n["y"], s=700, color=(0.9, 0.1, 0.1, min(0.35, n["infection"] * 0.35)), lw=0, zorder=0)\n        if n["type"] in _TYPE_STYLE:\n            c, m, _ = _TYPE_STYLE[n["type"]]\n            if n["type"] in ("factory", "generator") and not n.get("powered", True):\n                c = "#9e9e9e"\n            ax.scatter(n["x"], n["y"], s=170, color=c, marker=m, edgecolor="k", zorder=2)\n        ax.annotate(str(n["id"]), (n["x"], n["y"]), xytext=(5, -10), textcoords="offset points", fontsize=7, color="#777")\n    mine = set(st.you.get("cargo", []))\n    for t in st.tasks.values():\n        if t["status"] not in ("open", "announced") and t["id"] not in mine:\n            continue\n        o = st.nodes[t["origin"]]\n        c = "#43a047" if t["type"] == "rescue" else ("#fbc02d" if t.get("batch", -1) >= 0 else "#fb8c00")\n        if t["type"] == "fuel":\n            c = "#e53935"\n        ax.scatter(o["x"] + 0.7, o["y"] + 0.7, marker="^" if t["type"] == "rescue" else "s", s=45, color=c,\n                   alpha=1 if t["status"] == "open" else 0.35, edgecolor="k", lw=0.5, zorder=3)\n    for t in (st.tasks[i] for i in mine if i in st.tasks):\n        d = st.nodes[t["dest"]]\n        ax.scatter(d["x"], d["y"], marker="x", s=120, color="k", zorder=3)\n    for z in st.zombies:\n        ax.scatter(z["x"], z["y"], s=35, color="#c62828" if z["mode"] == "chase" else "#558b2f", marker="o", zorder=4)\n    e = st.escape\n    if e.get("enabled"):\n        ex = st.nodes[e["extraction"]]\n        ax.scatter(ex["x"], ex["y"], marker="*", s=700, color="#ffd54f", edgecolor="k", zorder=4)\n        ax.annotate("撤離點", (ex["x"], ex["y"]), xytext=(0, -22), textcoords="offset points", ha="center", fontsize=9, zorder=6)\n        ax.add_patch(plt.Circle((e["zone_x"], e["zone_y"]), e["zone_radius"], fill=False, ls="--", lw=2, color="#e53935", zorder=1))\n    for p in st.players:\n        if p["state"] == "eliminated":\n            continue\n        me = p["id"] == st.you.get("id")\n        ax.scatter(p["x"], p["y"], s=300 if me else 90, color=p["color"], edgecolor="k", lw=2.5 if me else 0.8, zorder=5,\n                   alpha=0.5 if p["state"] == "downed" else 1)\n        if me:\n            ax.annotate(f"{p[\'name\']} {p[\'score\']}", (p["x"], p["y"]), xytext=(0, 12), textcoords="offset points",\n                        ha="center", fontsize=10, fontweight="bold", zorder=6)\n    y = st.you\n    if e.get("enabled"):\n        if e["depart_at"] > 0:\n            goal = f"直升機 {max(0, int(e[\'depart_at\'] - st.time))} 秒後起飛"\n        elif e.get("ready_at", -1) >= 0:\n            goal = f"油桶 {e[\'fuel\']}/{e[\'fuel_required\']}（倒地就無法復活）"\n        else:\n            goal = f"零件 {e[\'parts\']}/{e[\'parts_required\']}"\n    else:\n        goal = f"城市感染 {st.city_infection:.0f}%"\n    ax.set_xlim(min(n["x"] for n in st.nodes.values()) - 3, max(n["x"] for n in st.nodes.values()) + 3)\n    ax.set_ylim(min(n["y"] for n in st.nodes.values()) - 3, max(n["y"] for n in st.nodes.values()) + 3)\n    ax.set_title(title or f"{st.time:4.0f}s / {st.raw[\'duration\']:.0f}s\u3000{goal}\u3000｜\u3000你：{y.get(\'score\', 0)} 分\u3000"\n                          f"生命 {y.get(\'hp\', 0):.0f}", fontsize=11)\n    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])\n    if show:\n        plt.show()\n\n_REPLAY_HTML = """\n<div id="RID" style="font-family:sans-serif;background:#141821;color:#e8e8e8;padding:10px;border-radius:10px;width:780px">\n<canvas width="760" height="520" style="background:#1d2230;border-radius:8px;display:block"></canvas>\n<div style="display:flex;align-items:center;gap:10px;margin-top:8px">\n<button class="pp" style="width:60px;font-size:15px">⏸</button>\n<input class="sl" type="range" min="0" value="0" style="flex:1">\n<span class="tm" style="width:110px;text-align:right"></span>\n<select class="sp"><option value="1">1x</option><option value="2">2x</option><option value="4" selected>4x</option><option value="8">8x</option><option value="16">16x</option></select>\n</div>\n<div class="hd" style="margin-top:6px;font-size:14px"></div>\n</div>\n<script>\n(function(){\nconst D = DATA, ME = MEID;\nconst root = document.getElementById("RID"), cv = root.querySelector("canvas"), cx = cv.getContext("2d");\nconst H = D.header, F = D.frames, every = H.every, N = H.nodes;\nconst xs = N.map(n => n[0]), ys = N.map(n => n[1]);\nconst x0 = Math.min(...xs) - 1.5, x1 = Math.max(...xs) + 1.5, y0 = Math.min(...ys) - 1.5, y1 = Math.max(...ys) + 1.5;\nconst sc = Math.min(760 / (x1 - x0), 520 / (y1 - y0));\nconst ox = (760 - (x1 - x0) * sc) / 2, oy = (520 - (y1 - y0) * sc) / 2;\nconst X = x => ox + (x - x0) * sc, Y = y => 520 - (oy + (y - y0) * sc);\nconst sl = root.querySelector(".sl"), pp = root.querySelector(".pp"), sp = root.querySelector(".sp");\nconst tm = root.querySelector(".tm"), hd = root.querySelector(".hd");\nsl.max = (F.length - 1) * 100;\nlet T = 0, playing = true, last = null;\nconst FAC = {shelter: "#ffd54f", hospital: "#f06292", supply: "#ffeb3b", factory: "#a1887f", generator: "#26c6da"};\nconst TASK = {p: "#ff9800", r: "#66bb6a", f: "#e53935", d: "#ff9800"};\nconst clock = t => { t = Math.max(0, t); return Math.floor(t / 60) + ":" + String(Math.floor(t % 60)).padStart(2, "0"); };\nfunction draw(fi) {\n  const i = Math.min(F.length - 1, Math.floor(fi)), f = F[i], g = F[Math.min(F.length - 1, i + 1)], a = fi - i;\n  cx.clearRect(0, 0, 760, 520);\n  const ex = N[H.extraction];\n  cx.setLineDash([6, 6]); cx.strokeStyle = "rgba(255,80,80,0.7)"; cx.lineWidth = 2;\n  cx.beginPath(); cx.arc(X(ex[0]), Y(ex[1]), f.z * sc, 0, 7); cx.stroke(); cx.setLineDash([]);\n  const blocked = new Set(f.b);\n  H.edges.forEach((e, k) => {\n    const A = N[e[0]], B = N[e[1]];\n    cx.strokeStyle = blocked.has(k) ? "#e53935" : e[2] ? "#8d6e63" : "#4a5266";\n    cx.lineWidth = e[2] ? 4 : 2; cx.setLineDash(blocked.has(k) ? [4, 4] : []);\n    cx.beginPath(); cx.moveTo(X(A[0]), Y(A[1])); cx.lineTo(X(B[0]), Y(B[1])); cx.stroke();\n  });\n  cx.setLineDash([]);\n  const on = new Set(f.g);\n  N.forEach((n, k) => {\n    if (n[2] === "road") return;\n    let c = FAC[n[2]] || "#fff";\n    if (n[2] === "generator" && !on.has(k)) c = "#666";\n    cx.fillStyle = c; const r = k === H.extraction ? 9 : 6;\n    cx.fillRect(X(n[0]) - r, Y(n[1]) - r, r * 2, r * 2);\n  });\n  const piles = {};\n  f.k.forEach(t => {\n    const c = piles[t[0]] = (piles[t[0]] || 0) + 1, n = N[t[0]];\n    cx.globalAlpha = t[2] ? 0.35 : 1; cx.fillStyle = TASK[t[1]] || "#fff";\n    cx.fillRect(X(n[0]) + 6 + ((c - 1) % 4) * 6, Y(n[1]) - 12 - Math.floor((c - 1) / 4) * 6, 5, 5);\n  });\n  cx.globalAlpha = 1;\n  f.i.forEach(e => {\n    const n = N[e[0]], r = 12 + 3 * Math.sin(T / 20);\n    cx.strokeStyle = "#ffab40"; cx.lineWidth = 2; cx.beginPath(); cx.arc(X(n[0]), Y(n[1]), r, 0, 7); cx.stroke();\n  });\n  f.zb.forEach(z => { cx.fillStyle = z[2] ? "#ff5252" : "#7cb342"; cx.beginPath(); cx.arc(X(z[0]), Y(z[1]), 3, 0, 7); cx.fill(); });\n  const nxt = {}; g.p.forEach(p => nxt[p[0]] = p);\n  let me = null;\n  f.p.forEach(p => {\n    if (p[3] === 2) return;\n    const q = nxt[p[0]] || p, px = p[1] + (q[1] - p[1]) * a, py = p[2] + (q[2] - p[2]) * a;\n    const info = H.players[p[0]] || ["?", "#fff"], mine = p[0] === ME;\n    if (mine) me = p;\n    cx.fillStyle = p[3] === 1 ? "#777" : info[1];\n    cx.beginPath(); cx.arc(X(px), Y(py), mine ? 8 : 5, 0, 7); cx.fill();\n    if (mine) { cx.strokeStyle = "#fff"; cx.lineWidth = 2.5; cx.stroke(); cx.fillStyle = "#fff"; cx.font = "13px sans-serif"; cx.fillText(info[0], X(px) + 10, Y(py) - 8); }\n  });\n  const goal = f.depart >= 0 ? "起飛倒數 " + clock(f.depart - f.t) : f.ready >= 0 ? "油桶 " + f.fuel + "/" + f.freq : "零件 " + f.parts + "/" + f.preq;\n  hd.textContent = goal + (me ? "\u3000｜\u3000你：" + me[4] + " 分\u3000生命 " + me[5] : "");\n  tm.textContent = clock(f.t) + " / " + clock(H.duration);\n}\nfunction tick(now) {\n  if (last !== null && playing) { T += (now - last) / 1000 / every * sp.value; if (T >= F.length - 1) { T = F.length - 1; playing = false; pp.textContent = "▶"; } sl.value = T * 100; }\n  last = now; draw(T); requestAnimationFrame(tick);\n}\npp.onclick = () => { if (T >= F.length - 1) T = 0; playing = !playing; pp.textContent = playing ? "⏸" : "▶"; };\nsl.oninput = () => { T = sl.value / 100; };\nrequestAnimationFrame(tick);\n})();\n</script>\n"""\n\n\ndef replay(game, name: str | None = None):\n    import json, random\n    from IPython.display import HTML, display\n    if isinstance(game, dict):\n        data = game\n    else:\n        data = game.replay_data()\n        name = name or game.me.name\n    if not data.get("frames"):\n        print("沒有可以重播的畫面")\n        return\n    players = data["header"]["players"]\n    me = next((int(k) for k, v in players.items() if name and v[0] == name), -1)\n    if name and me < 0:\n        print(f"重播裡找不到「{name}」，照樣播放（名字要和加入時一樣）")\n    rid = "rp" + str(random.randrange(10 ** 9))\n    html = (_REPLAY_HTML.replace("RID", rid).replace("MEID", str(me))\n            .replace("DATA", json.dumps(data, ensure_ascii=False, separators=(",", ":"))))\n    display(HTML(html))\n\n\ndef replay_match(url: str, name: str | None = None):\n    data = Game(url, timeout=120)._req("GET", "/replay")\n    replay(data, name)\n    return data\n\ndef benchmark(url: str, make_policy: Callable[[], Callable], matches: int = 8, name: str = "Bench",\n              token: str | None = None, interval: float = 1.0) -> list[dict]:\n    game = Game(url, token=token)\n    if token is None:\n        game.join(name)\n    me_id = game.state().you.get("id", -1)\n    rows, last = [], None\n    for k in range(matches):\n        st = game.wait_until_running(after_match=last)\n        last = st.raw.get("match_id")\n        late = st.time > 10\n        game.run(make_policy(), interval=interval, verbose=False)\n        res = {}\n        for _ in range(20):\n            res = game.results()\n            if res.get("match_id") == last:\n                break\n            time.sleep(0.5)\n        ranking = res.get("ranking", [])\n        me = next((r for r in ranking if r["player_id"] == me_id), None)\n        if me is None:\n            print(f"第 {k + 1} 場：找不到結算")\n            continue\n        row = dict(match=k + 1, seed=res["seed"], score=me["score"], rank=me["rank"], players=len(ranking),\n                   escaped=me.get("escaped", False), late=late)\n        rows.append(row)\n        print(f"第 {k + 1:2d} 場 種子 {row[\'seed\']:3d}：{row[\'score\']:5d} 分 第 {row[\'rank\']}/{row[\'players\']} 名  "\n              f"{\'搭上直升機\' if row[\'escaped\'] else \'沒搭上\'}{\'  （中途加入，不計）\' if late else \'\'}")\n    counted = [r for r in rows if not r["late"]]\n    if counted:\n        avg = sum(r["score"] for r in counted) / len(counted)\n        rank = sum(r["rank"] for r in counted) / len(counted)\n        print(f"\\n總結（{len(counted)} 場）：平均 {avg:.0f} 分，平均第 {rank:.1f} 名")\n    return rows\n', 'apocalypse_local.py': 'from __future__ import annotations\n\nimport math\nimport time as _time\n\nimport numpy as np\n\nINF = 1e9\nF03, F05, F06, F07 = (float(np.float32(x)) for x in (0.3, 0.5, 0.6, 0.7))\nPHASE_KEYS = ["early", "mid", "late", "final"]\nPHASE_ENDS = [0.25, 0.5833, 0.8333, 1.0]\nPALETTE = ["#4DC2F7", "#FF7042", "#BA68C7", "#FFD64F", "#66BA6B", "#F06191", "#4FD1C4", "#FA9E21", "#9EA8FF", "#CCDB38"]\n\nDEFAULTS = dict(\n    mapCols=10, mapRows=8, bridges=2,\n    batchChance=0.6, batchMin=3, batchMax=5, batchBonusRatio=1.0, taskInterval=9.0, announceLead=30.0,\n    seed=42, matchDuration=600.0, randomBots=0, greedyBots=0, botReactionTime=1.0,\n    planCooldown=15.0, maxPlanSteps=200, capacity=10, playerSpeed=3.0, loadSpeedPenalty=0.05, fuelWeight=4, weaponWeight=4, weaponShots=8, weaponStock=2, killReward=5, weaponRange=3.5, weaponInterval=1.2,\n    downedTime=15.0, survivalBonus=120, reviveBonus=60, respawnPenalty=50, respawnHp=70.0,\n    cityRate=0.55, spreadAmount=0.15, spreadGrow=0.02,\n\n    coop=True, coopCityRate=1.35, cityPerPlayer=0.02, cityPerPoint=0.03, classBonus=200, urgentFailCity=3.0, rescueFailCity=2.0,\n    coopTaskPlayers=3.0,\n\n    escape=True, partsBase=0.0, partsPerPlayer=7.0, partsCrowding=0.15, difficulty=1.2, earliestDepart=480.0, departDelay=60.0, escapeBonus=200, earlyBonusPerSec=1.0,\n    zoneStart=90.0, zoneEnd=540.0, zoneMinRadius=7.0, zoneDamage=3.0, zoneInfection=1.0,\n\n    incidentInterval=30.0, roadblockInterval=45.0, roadblockTime=60.0, civilianZombies=1, civiliansPerPlayer=2.5, trappedCivilians=4, generatorWeight=0.04, generatorCooldown=90.0, speedBoost=1.2, speedBoostTime=60.0, guardDownTime=60.0,\n    guardRange=5.0, guardFireInterval=1.0,\n\n    fuelPerPlayer=0.6, fuelSpare=4, fuelSparePerPlayer=0.3, fuelReward=80, boardTime=45.0,\n\n    stunTime=40.0,\n\n    partWeightReward=12.0, rescueRewardMul=2.0, incidentRewardMul=2.0,\n\n    partDecay=80.0,\n    comboWindow=25.0, comboStep=10, comboMax=5,\n    zombieSpeed=1.4, zombieSpeedPerPhase=0.2, zombieSight=5.5, zombieAttackRange=0.9,\n    zombieDamage=5.5, zombieInfection=2.5, zombieChaseMul=1.25, zombieCapBase=6.0, zombieCapGrowth=40.0, zombieSpawnMul=1.6,\n    actionNoiseRadius=10.0, actionNoiseDuration=12.0, survivorNoiseRadius=8.0, screamRadius=12.0,\n)\n\ndef _i32(x):\n    x &= 0xFFFFFFFF\n    return x - (1 << 32) if x >= 1 << 31 else x\n\nclass DotNetRandom:\n    MBIG, MSEED = 2147483647, 161803398\n\n    def __init__(self, seed: int):\n        seed = _i32(seed)\n        sub = self.MBIG if seed == -2147483648 else abs(seed)\n        a = [0] * 56\n        mj = _i32(self.MSEED - sub)\n        a[55] = mj\n        mk = 1\n        for i in range(1, 55):\n            ii = (21 * i) % 55\n            a[ii] = mk\n            mk = _i32(mj - mk)\n            if mk < 0:\n                mk += self.MBIG\n            mj = a[ii]\n        for _ in range(4):\n            for i in range(1, 56):\n                a[i] = _i32(a[i] - a[1 + (i + 30) % 55])\n                if a[i] < 0:\n                    a[i] = _i32(a[i] + self.MBIG)\n        self.a, self.inext, self.inextp = a, 0, 21\n\n    def _sample(self):\n        i = self.inext + 1\n        if i >= 56:\n            i = 1\n        j = self.inextp + 1\n        if j >= 56:\n            j = 1\n        r = _i32(self.a[i] - self.a[j])\n        if r == self.MBIG:\n            r -= 1\n        if r < 0:\n            r = _i32(r + self.MBIG)\n        self.a[i] = r\n        self.inext, self.inextp = i, j\n        return r\n\n    def next(self, n=None):\n        if n is None:\n            return self._sample()\n        return int(self._sample() * (1.0 / self.MBIG) * n)\n\n    def next_double(self):\n        return self._sample() * (1.0 / self.MBIG)\n\ndef _f32(x):\n    return float(np.float32(x))\n\ndef _dist(a, b):\n    dx, dy = np.float32(a[0]) - np.float32(b[0]), np.float32(a[1]) - np.float32(b[1])\n    return float(np.float32(math.sqrt(float(dx * dx + dy * dy))))\n\ndef _order_by_random(items, rng):\n    keys = [rng.next() for _ in items]\n    return [items[i] for i in sorted(range(len(items)), key=lambda i: keys[i])]\n\ndef _round(x):\n    return int(round(x))\n\nclass Node:\n    __slots__ = ("id", "type", "pos", "infection", "medkits", "edges", "district", "powered", "weapons")\n\n    def __init__(self, id, pos):\n        self.id, self.type, self.pos, self.infection, self.medkits, self.edges = id, "road", pos, 0.0, 0, []\n        self.weapons = 0\n        self.powered = False\n        self.district = "city"\n\n    @property\n    def safe(self):\n        return self.type in ("shelter", "hospital")\n\nclass Edge:\n    __slots__ = ("id", "a", "b", "length", "bridge", "blocked", "blocked_until")\n\n    def __init__(self, id, a, b, length, bridge):\n        self.id, self.a, self.b, self.length, self.bridge = id, a, b, length, bridge\n        self.blocked, self.blocked_until = False, 0.0\n\n    def other(self, n):\n        return self.b if n == self.a else self.a\n\nDIAG_DIV, REMOVE_DIV, KEEP_DEG = 14, 1, 2\n\ndef _sign(x):\n    return (x > 0) - (x < 0)\n\nclass CityMap:\n    spacing = 6.0\n\n    def __init__(self, cols, rows, bridges):\n        self.cols, self.rows = max(4, cols), max(3, rows)\n        self.bridge_count = min(max(bridges, 1), self.rows)\n        self.river: list[tuple[int, int]] = []\n        self.river_set: set = set()\n        self.cuts: set = set()\n        self.side_a: list[bool] = []\n        self.warp_x: list = []\n        self.warp_y: list = []\n        self.warp_w = self.warp_h = 0\n        self.nodes: list[Node] = []\n        self.edges: list[Edge] = []\n        self.start_node = 0\n\n    @property\n    def scale(self):\n        return len(self.nodes) / 48.0\n\n    def west(self, node):\n        return self.side_a[node]\n\n    def warp(self, gx, gy):\n        f = np.float32\n        u, v = f(gx) / f(3), f(gy) / f(3)\n        i = min(max(int(math.floor(u)), 0), self.warp_w - 2)\n        j = min(max(int(math.floor(v)), 0), self.warp_h - 2)\n        fu, fv = u - f(i), v - f(j)\n        W = self.warp_w\n\n        def lerp2(w):\n            a = w[j * W + i] + (w[j * W + i + 1] - w[j * W + i]) * fu\n            b = w[(j + 1) * W + i] + (w[(j + 1) * W + i + 1] - w[(j + 1) * W + i]) * fu\n            return a + (b - a) * fv\n        return lerp2(self.warp_x), lerp2(self.warp_y)\n\n    def river_in_cell(self, r, c):\n        return (c + 1, r + 1) in self.river_set\n\n    @staticmethod\n    def _key(a, b):\n        return min(a, b) * 10000 + max(a, b)\n\n    def _cut(self, a, b):\n        return self._key(a, b) in self.cuts\n\n    def find_edge(self, a, b):\n        for e in self.nodes[a].edges:\n            if self.edges[e].other(a) == b:\n                return self.edges[e]\n        return None\n\n    def _entry(self, side, rng):\n        cols, rows = self.cols, self.rows\n        if side == 0:\n            return (2 + rng.next(cols - 3), 0)\n        if side == 1:\n            return (cols, 2 + rng.next(rows - 3))\n        if side == 2:\n            return (2 + rng.next(cols - 3), rows)\n        return (0, 2 + rng.next(rows - 3))\n\n    def _inward(self, p):\n        return (min(max(p[0], 1), self.cols - 1), min(max(p[1], 1), self.rows - 1))\n\n    @staticmethod\n    def _stair(path, to, rng):\n        p = path[-1]\n        while p != to:\n            dx, dy = _sign(to[0] - p[0]), _sign(to[1] - p[1])\n            if dx != 0 and dy != 0:\n                if rng.next(2) == 0:\n                    dy = 0\n                else:\n                    dx = 0\n            p = (p[0] + dx, p[1] + dy)\n            path.append(p)\n\n    def _step_edge(self, p, q):\n        cols = self.cols\n        if p[1] == q[1]:\n            x = min(p[0], q[0])\n            return self._key((p[1] - 1) * cols + x, p[1] * cols + x)\n        y = min(p[1], q[1])\n        return self._key(y * cols + p[0] - 1, y * cols + p[0])\n\n    def _split(self, path):\n        cols, rows = self.cols, self.rows\n        self.cuts = {self._step_edge(path[i], path[i + 1]) for i in range(len(path) - 1)}\n        side = [False] * (rows * cols)\n        side[0] = True\n        queue = [0]\n        while queue:\n            u = queue.pop(0)\n            r, c = u // cols, u % cols\n            for v in (u - 1 if c > 0 else -1, u + 1 if c + 1 < cols else -1, u - cols if r > 0 else -1, u + cols if r + 1 < rows else -1):\n                if v >= 0 and not side[v] and not self._cut(u, v):\n                    side[v] = True\n                    queue.append(v)\n        self.side_a = side\n\n    def _generate_river(self, rng):\n        cols, rows = self.cols, self.rows\n        n, best_gap = rows * cols, None\n        best = [(cols // 2, y) for y in range(rows + 1)]\n        for _ in range(80):\n            s0 = rng.next(4)\n            s1 = (s0 + 1 + rng.next(3)) % 4\n            a = self._entry(s0, rng)\n            b = self._entry(s1, rng)\n            mid = (2 + rng.next(cols - 3), 2 + rng.next(rows - 3))\n            path = [a, self._inward(a)]\n            self._stair(path, mid, rng)\n            self._stair(path, self._inward(b), rng)\n            path.append(b)\n            if len(set(path)) != len(path):\n                continue\n            self._split(path)\n            gap = abs(sum(self.side_a) * 2 - n)\n            if best_gap is None or gap < best_gap:\n                best_gap, best = gap, path\n            if gap * 10 <= n:\n                break\n        self._split(best)\n        self.river = best\n        self.river_set = set(best)\n\n    def generate(self, rng):\n        cols, rows = self.cols, self.rows\n        f = np.float32\n        self.warp_w, self.warp_h = cols // 3 + 2, rows // 3 + 2\n        self.warp_x, self.warp_y = [], []\n        for _ in range(self.warp_w * self.warp_h):\n            self.warp_x.append(f(rng.next_double() - 0.5) * f(5))\n            self.warp_y.append(f(rng.next_double() - 0.5) * f(5))\n        sx, sy = f((cols - 1) * self.spacing), f((rows - 1) * self.spacing)\n        for r in range(rows):\n            for c in range(cols):\n                jx = f(rng.next_double() - 0.5) * f(2.2)\n                jy = f(rng.next_double() - 0.5) * f(2.2)\n                wx, wy = self.warp(c, r)\n                x = min(max(f(c * self.spacing) + jx + wx, f(-2.5)), sx + f(2.5))\n                y = min(max(f(r * self.spacing) + jy + wy, f(-2.5)), sy + f(2.5))\n                self.nodes.append(Node(r * cols + c, (_f32(x), _f32(y))))\n        self._generate_river(rng)\n        steps = [self._step_edge(self.river[i], self.river[i + 1]) for i in range(len(self.river) - 1)]\n        third = max(1, len(steps) // 3)\n        bridge_keys = {steps[rng.next(third)]}\n        bridge_keys.add(steps[len(steps) - 1 - rng.next(third)])\n        if self.bridge_count > 2:\n            for k in _order_by_random([k for k in steps if k not in bridge_keys], rng)[:self.bridge_count - 2]:\n                bridge_keys.add(k)\n        for r in range(rows):\n            for c in range(cols):\n                i = r * cols + c\n                if c + 1 < cols:\n                    if not self._cut(i, i + 1):\n                        self._add(i, i + 1, False)\n                    elif self._key(i, i + 1) in bridge_keys:\n                        self._add(i, i + 1, True)\n                if r + 1 < rows:\n                    if not self._cut(i, i + cols):\n                        self._add(i, i + cols, False)\n                    elif self._key(i, i + cols) in bridge_keys:\n                        self._add(i, i + cols, True)\n        for _ in range(cols * rows // DIAG_DIV):\n            c, r = rng.next(cols - 1), rng.next(rows - 1)\n            down = rng.next(2) == 0\n            if self.river_in_cell(r, c):\n                continue\n            if down:\n                self._add(r * cols + c, (r + 1) * cols + c + 1, False)\n            else:\n                self._add(r * cols + c + 1, (r + 1) * cols + c, False)\n        cands = _order_by_random([e for e in self.edges if not e.bridge], rng)[:len(self.edges) // REMOVE_DIV]\n        for e in cands:\n            if len(self.nodes[e.a].edges) <= KEEP_DEG or len(self.nodes[e.b].edges) <= KEEP_DEG:\n                continue\n            e.blocked = True\n            if not self._reachable():\n                e.blocked = False\n                continue\n            self._remove(e)\n        self._assign_types(rng)\n        self.recompute()\n\n    def _add(self, a, b, bridge):\n        old = self.find_edge(a, b)\n        if old:\n            return old\n        e = Edge(len(self.edges), a, b, _dist(self.nodes[a].pos, self.nodes[b].pos), bridge)\n        self.edges.append(e)\n        self.nodes[a].edges.append(e.id)\n        self.nodes[b].edges.append(e.id)\n        return e\n\n    def _remove(self, e):\n        self.edges.remove(e)\n        for n in self.nodes:\n            n.edges.clear()\n        for i, ed in enumerate(self.edges):\n            ed.id = i\n            self.nodes[ed.a].edges.append(i)\n            self.nodes[ed.b].edges.append(i)\n\n    def _assign_types(self, rng):\n        cols, rows = self.cols, self.rows\n        for s in (( rows - 1) * cols, cols - 1):\n            self.nodes[s].infection = 1.0\n            for e in self.nodes[s].edges:\n                self.nodes[self.edges[e].other(s)].infection = 0.5\n\n        def free(n):\n            return n.type == "road" and n.infection == 0\n        left = _order_by_random([n for n in self.nodes if free(n) and self.west(n.id)], rng)\n        right = _order_by_random([n for n in self.nodes if free(n) and not self.west(n.id)], rng)\n\n        def take(pool, t, count):\n            for _ in range(count):\n                if not pool:\n                    return\n                pool.pop(0).type = t\n        extra = max(0, _round((rows * cols / 48.0 - 1.0) * 2.0))\n        take(left, "shelter", 2 + extra // 2); take(left, "hospital", 1); take(left, "supply", 2); take(left, "factory", 1 + (extra + 1) // 2); take(left, "generator", 1)\n        take(right, "shelter", 1 + extra // 2); take(right, "hospital", 1); take(right, "supply", 1 + extra // 2); take(right, "factory", 2 + (extra + 1) // 2); take(right, "generator", 1)\n        take(left if rng.next(2) == 0 else right, "armory", 1)\n        for n in self.nodes:\n            if n.type == "supply":\n                n.medkits = 3\n        cx, cy = (cols - 1) * self.spacing * 0.5, (rows - 1) * self.spacing * 0.5\n        shelters = [n for n in self.nodes if n.type == "shelter"]\n        self.start_node = min(shelters, key=lambda n: _dist(n.pos, (cx, cy))).id\n\n    def _reachable(self):\n        seen = [False] * len(self.nodes)\n        seen[0] = True\n        queue, count = [0], 1\n        while queue:\n            u = queue.pop()\n            for i in self.nodes[u].edges:\n                e = self.edges[i]\n                if e.blocked:\n                    continue\n                v = e.other(u)\n                if not seen[v]:\n                    seen[v] = True\n                    count += 1\n                    queue.append(v)\n        return count == len(self.nodes)\n\n    def connected(self):\n        return bool(np.all(self.dist[0] < INF))\n\n    def recompute(self):\n        self.dist, self.next = self._floyd()\n\n    def _floyd(self):\n        n = len(self.nodes)\n        d = np.full((n, n), np.float32(INF), dtype=np.float32)\n        nx = np.full((n, n), -1, dtype=np.int64)\n        np.fill_diagonal(d, 0)\n        nx[np.arange(n), np.arange(n)] = np.arange(n)\n        for e in self.edges:\n            if e.blocked:\n                continue\n            d[e.a, e.b] = d[e.b, e.a] = e.length\n            nx[e.a, e.b], nx[e.b, e.a] = e.b, e.a\n        for k in range(n):\n            reach = d[:, k] < INF\n            alt = d[:, k:k + 1] + d[k:k + 1, :]\n            better = (alt < d) & reach[:, None]\n            d = np.where(better, alt, d)\n            nx = np.where(better, nx[:, k:k + 1], nx)\n        return d, nx\n\n    def path(self, a, b, nxt=None):\n        nxt = self.next if nxt is None else nxt\n        if nxt[a, b] < 0:\n            return None\n        out, u = [], a\n        while u != b:\n            u = int(nxt[u, b])\n            out.append(u)\n        return out\n\nclass Task:\n    def __init__(self, **kw):\n        self.bonus, self.urgent, self.deadline, self.expires_at = 0, False, -1.0, -1.0\n        self.carrier, self.completed_by, self.batch = -1, -1, -1\n        self.__dict__.update(kw)\n\nclass Batch:\n    def __init__(self, id, origin, deadline):\n        self.id, self.origin, self.deadline = id, origin, deadline\n        self.bonus, self.owner, self.delivered, self.paid, self.failed, self.tasks = 0, -1, 0, False, False, []\n        self.by = {}\n\nclass Player:\n    def __init__(self, id, name, color, bot=None, bot_kind=""):\n        self.id, self.name, self.color, self.bot, self.bot_kind = id, name, color, bot, bot_kind\n        self.team = id\n        self.node, self.next_node, self.progress, self.pos = 0, -1, 0.0, (0.0, 0.0)\n        self.hp, self.infection, self.cargo, self.medkits = 100.0, 0.0, [], 0\n        self.state, self.downed_timer, self.eliminated_at, self.eliminated_reason = "active", 0.0, -1.0, ""\n        self.score = self.delivered = self.rescued = self.revives = self.downs = 0\n        self.evacuated = False\n        self.city_reduced = 0.0\n        self.plan, self.plan_index, self.pending, self.action_timer = [], 0, None, 0.0\n        self.last_error, self.last_plan_time, self.idle_since = "", -999.0, -1.0\n        self.noise_timer, self.last_scream, self.rank = 0.0, -999.0, 0\n        self.role = ""\n        self.fixed = 0\n        self.fuel = 0\n        self.parts = 0\n        self.escaped = False\n        self.combo, self.last_score_at = 0, -999.0\n        self.protect_until = -1.0\n        self.revived_at = -999.0\n        self.ammo, self.kills, self.fire_cooldown = 0, 0, 0.0\n\nclass Zombie:\n    def __init__(self, id, node, pos, from_player, rng):\n        self.id, self.node, self.pos, self.from_player, self.rng = id, node, pos, from_player, rng\n        self.next_node, self.progress, self.wait = -1, 0.0, 1.0\n        self.mode, self.target, self.chasing = "wander", -1, -1\n\nclass Noise:\n    def __init__(self, id, node, pos, radius, until, source):\n        self.id, self.node, self.pos, self.radius, self.until, self.source = id, node, pos, radius, until, source\n\ndef _lerp(a, b, t):\n    return (a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t)\n\ndef _vd(a, b):\n    return math.hypot(a[0] - b[0], a[1] - b[1])\n\ndef _step(action, node=-1, task=-1, target=-1, seconds=0.0):\n    return {"action": action, "node": node, "task": task, "target": target, "seconds": seconds}\n\ndef _heal_plan(sim, p):\n    if p.medkits > 0:\n        return [_step("use_medkit")]\n    hs = [n for n in sim.map.nodes if n.type == "hospital"]\n    if not hs:\n        return []\n    h = min(hs, key=lambda n: sim.map.dist[p.node, n.id])\n    return [_step("goto", node=h.id), _step("heal")]\n\ndef _deliver_plan(sim, p):\n    out = []\n    for t in sorted([sim.task_by_id(i) for i in p.cargo if sim.task_by_id(i)], key=lambda t: sim.map.dist[p.node, t.dest]):\n        out += [_step("goto", node=t.dest), _step("dropoff", task=t.id)]\n    return out\n\ndef _do_task(t):\n    return [_step("goto", node=t.origin), _step("pickup", task=t.id), _step("goto", node=t.dest), _step("dropoff", task=t.id)]\n\nclass RandomBot:\n    def __init__(self, seed):\n        self.rng = DotNetRandom(_i32(seed * 7919 + 1))\n\n    def think(self, sim, p):\n        if p.hp < 30 or p.infection > 70:\n            return _heal_plan(sim, p)\n        if p.cargo:\n            return _deliver_plan(sim, p)\n        opn = [t for t in sim.tasks if t.status == "open" and sim.can_take(p, t) and sim.load(p) + t.weight <= sim.cfg["capacity"]]\n        if not opn:\n            return [_step("wait", seconds=2)]\n        return _do_task(opn[self.rng.next(len(opn))])\n\nclass GreedyBot:\n    def think(self, sim, p):\n        m = sim.map\n        if p.hp < 45 or p.infection > 55:\n            return _heal_plan(sim, p)\n        if p.cargo:\n            return _deliver_plan(sim, p)\n        steps, frm = [], p.node\n        if p.medkits == 0:\n            sups = [n for n in m.nodes if n.type == "supply" and n.medkits > 0]\n            if sups:\n                s = min(sups, key=lambda n: m.dist[p.node, n.id])\n                if m.dist[p.node, s.id] < 10:\n                    steps += [_step("goto", node=s.id), _step("take_medkit")]\n                    frm = s.id\n        best, best_v = None, 0.0\n        load = sim.load(p) + (1 if steps else 0)\n        for t in sim.tasks:\n            if t.status != "open" or not sim.can_take(p, t) or load + t.weight > sim.cfg["capacity"]:\n                continue\n            travel = float(m.dist[frm, t.origin] + m.dist[t.origin, t.dest])\n            if travel >= INF:\n                continue\n            eta = sim.time + travel / sim.cfg["playerSpeed"] + 2\n            gain = t.reward + (t.bonus if t.urgent and eta <= t.deadline else 0)\n            v = gain * (1 - 0.5 * m.nodes[t.origin].infection) / (travel + 5)\n            if v > best_v:\n                best, best_v = t, v\n        if best is None:\n            return steps + [_step("wait", seconds=2)]\n        return steps + _do_task(best)\n\nACTIONS = {"move", "goto", "pickup", "dropoff", "take_medkit", "use_medkit", "heal", "revive", "wait", "fix", "take_weapon", "drop", "drop_medkit", "drop_weapon"}\nEMERGENCY = {"move", "goto", "wait", "use_medkit", "heal", "drop", "drop_medkit", "drop_weapon"}\nDROPS = {"drop", "drop_medkit", "drop_weapon"}\nS_TASK, S_EVENT, S_SPREAD, S_ZOMBIE, S_BLOCK = 1, 2, 3, 4, 5\n\nINCIDENTS = {\n    "outbreak": ("感染爆發", 8.0, 1, 60.0, 60),\n    "bridge": ("橋樑損毀", 10.0, 1, 9999.0, 50),\n    "airdrop": ("物資空投", 4.0, 1, 45.0, 40),\n    "trapped": ("受困人群", 20.0, 2, 75.0, 120),\n    "guard": ("撤離點告急", 8.0, 1, 50.0, 60),\n    "heli": ("直升機故障", 24.0, 2, 60.0, 80),\n    "medevac": ("醫療隊抵達", 6.0, 1, 50.0, 80),\n    "napalm": ("呼叫空襲", 8.0, 1, 50.0, 60),\n    "decoy": ("聲波誘餌", 20.0, 2, 60.0, 90),\n    "chest": ("神秘寶箱", 3.0, 1, 40.0, 150),\n    "boss": ("殭屍王", 30.0, 3, 75.0, 200),\n\n    "herbs": ("草藥採集", 6.0, 1, 50.0, 70),\n    "gas": ("加油站", 5.0, 1, 60.0, 80),\n    "rubble": ("瓦礫堆", 20.0, 2, 70.0, 100),\n    "bell": ("教堂鐘聲", 8.0, 1, 50.0, 70),\n\n    "generator": ("發電機故障", 15.0, 1, 9999.0, 60),\n}\n\nDISTRICTS = ["jungle", "city", "ruins", "town"]\nDISTRICT_NAMES = {"jungle": "雨林", "city": "城市", "ruins": "廢墟", "town": "小城鎮"}\nDISTRICT_EVENT = {"jungle": "herbs", "city": "gas", "ruins": "rubble", "town": "bell"}\n\nINCIDENT_TEXT = {\n    "outbreak": ("壓制了感染爆發!", "感染爆發失控了..."),\n    "bridge": ("修好了斷掉的橋!", "橋沒修好..."),\n    "airdrop": ("搶到了空投物資!", "空投物資被搶走了..."),\n    "trapped": ("救出了受困的民眾!", "沒能救出受困的民眾..."),\n    "guard": ("守住了撤離點!", "撤離點失守了!"),\n    "heli": ("排除了直升機故障!", "直升機故障沒排除!"),\n    "medevac": ("接應到醫療隊!", "沒接應到醫療隊..."),\n    "napalm": ("呼叫的空襲命中了!", "空襲沒呼叫成功..."),\n    "decoy": ("啟動了聲波誘餌!", "聲波誘餌沒啟動..."),\n    "chest": ("打開了神秘寶箱!", "沒搶到神秘寶箱..."),\n    "boss": ("打倒了殭屍王!", "殭屍王沒被打倒..."),\n    "herbs": ("採到了草藥!", "草藥沒採集到..."),\n    "gas": ("裝滿了油桶!", "加油站的油被搶光了..."),\n    "rubble": ("挖開了瓦礫堆!", "瓦礫堆沒挖開!"),\n    "bell": ("敲響了教堂的鐘!", "教堂的鐘沒敲響!"),\n    "generator": ("修好了發電機!", "發電機沒修好"),\n}\n\nclass Incident:\n    def __init__(self, id, kind, node, time):\n        name, need, min_people, limit, reward = INCIDENTS[kind]\n        self.id, self.kind, self.name, self.node = id, kind, name, node\n        self.need, self.min_people, self.deadline, self.reward = need, min_people, time + limit, reward\n        self.progress, self.status, self.workers, self.edge = 0.0, "active", 0, -1\n        self.work = {}\n\nclass Sim:\n\n    def __init__(self, **cfg):\n        self.cfg = dict(DEFAULTS, **cfg)\n        c = self.cfg\n        self.rng = DotNetRandom(c["seed"])\n        self.map = CityMap(c["mapCols"], c["mapRows"], c["bridges"])\n        self.map.generate(self.rng)\n        self.status, self.time, self.city_infection, self.end_reason = "lobby", 0.0, 0.0, ""\n        self.class_won = False\n\n        m = self.map\n        if c["escape"]:\n            for n in m.nodes:\n                if n.type == "shelter" and n.id != m.start_node:\n                    n.type = "road"\n        self.extraction = m.start_node\n        drng = DotNetRandom(_i32(self.cfg["seed"] * 7919 + 17))\n        west, east = DISTRICTS[drng.next(4)], DISTRICTS[drng.next(4)]\n        for nd in m.nodes:\n            nd.district = west if m.west(nd.id) else east\n        for nd in m.nodes:\n            if nd.type == "armory":\n                nd.weapons = c["weaponStock"]\n        self.player_shots = []\n        self.parts, self.parts_required, self.ready_at, self.depart_at = 0, 0, -1.0, -1.0\n        self.guard_kills = 0\n        self.guard_cooldown, self.guard_shots = 0.0, []\n        self.incidents: list[Incident] = []\n        self.evacuated = 0\n        self.speed_boost_until = self.guard_down_until = -1.0\n        self.zombie_stun_until = -1.0\n        self.medevac_timer, self.medevac_next, self.medevac_rolls = 5.0, 0.0, 0\n        self.heli_boost = False\n        self.fuel, self.fuel_required, self.fuel_discount = 0, 0, 0\n        self.incident_timer = 25.0\n        self.roadblock_timer, self.roadblock_index = 40.0, 0\n        self.last_phase = 0\n        self.last_generator_break = -999.0\n        xs = [n.pos[0] for n in m.nodes]; ys = [n.pos[1] for n in m.nodes]\n        ex = m.nodes[self.extraction].pos\n        self.zone_max = max(math.hypot(x - ex[0], y - ex[1]) for x, y in zip(xs, ys)) + 1.0\n        self.players: list[Player] = []\n        self.tasks: list[Task] = []\n        self.zombies: list[Zombie] = []\n        self.events: list[dict] = []\n        self.noises: list[Noise] = []\n        self.batches: dict[int, Batch] = {}\n        self._ids = dict(task=0, zombie=0, player=0, batch=0, noise=0, incident=0)\n        self.task_timer = self.spread_timer = self.zombie_timer = self.restock_timer = self.bot_timer = 0.0\n        self.event_timer = 20.0\n        self.task_slot = self.event_index = self.spread_tick = self.zombie_spawn_index = 0\n\n    def _id(self, k):\n        v = self._ids[k]\n        self._ids[k] += 1\n        return v\n\n    def use_stream(self, stream, index):\n        c = self.cfg\n        self.rng = DotNetRandom(_i32(_i32(c["seed"] * 73856093) ^ _i32(stream * 19349663) ^ _i32((index + 1) * 83492791)))\n\n    def time_left(self):\n        c = self.cfg\n        return max(0.0, c["matchDuration"] - (self.depart_at - c["boardTime"] if c["escape"] and self.depart_at >= 0 else self.time))\n\n    @property\n    def phase(self):\n        f = self.time / self.cfg["matchDuration"]\n        for i, e in enumerate(PHASE_ENDS):\n            if f < e:\n                return i\n        return 3\n\n    @property\n    def task_scale(self):\n        if not self.cfg["coop"]:\n            return 1.0\n        return max(1.0, len(self.players) / self.cfg["coopTaskPlayers"])\n\n    @property\n    def zombie_speed(self):\n        return self.cfg["zombieSpeed"] + self.cfg["zombieSpeedPerPhase"] * self.phase\n\n    def log(self, type_, text, node=-1):\n        self.events.append({"time": self.time, "type": type_, "text": text, "node": node})\n        if len(self.events) > 200:\n            self.events.pop(0)\n\n    def add_player(self, name, bot=None, bot_kind=""):\n        pid = self._id("player")\n        p = Player(pid, name, PALETTE[pid % len(PALETTE)], bot, bot_kind)\n        p.node = self.map.start_node\n        p.pos = self.map.nodes[p.node].pos\n        self.players.append(p)\n        if self.cfg["escape"] and self.status == "running":\n            self.parts_required = max(self.parts_required, self.parts_needed(len(self.players)))\n        return p\n\n    def parts_needed(self, n):\n        c = self.cfg\n        n = min(n, 30)\n        return max(5, _round((c["partsBase"] + c["partsPerPlayer"] * n - c["partsCrowding"] * n * n) * c["difficulty"] * self.dscale))\n\n    @property\n    def dscale(self):\n        return self.cfg["matchDuration"] / 600.0\n\n    def start(self):\n        c = self.cfg\n        for i in range(c["randomBots"]):\n            self.add_player(f"Bot-Random-{i + 1}", RandomBot(c["seed"] + i * 31), "random")\n        for i in range(c["greedyBots"]):\n            self.add_player(f"Bot-Greedy-{i + 1}", GreedyBot(), "greedy")\n        self.status = "running"\n        if c["escape"]:\n            self.parts_required = self.parts_needed(len(self.players))\n            gens = self.generators()\n            for g in gens:\n                self._break_generator(g)\n            self._spawn_civilians()\n            self.log("incident", "發電機都沒電！先修好發電機，工廠才會生產零件", gens[0].id if gens else -1)\n        for _ in range(_round(6 * self.map.scale * self.task_scale)):\n            self.use_stream(S_TASK, self.task_slot); self.task_slot += 1\n            self.spawn_task(False, 0.0)\n        for _ in range(_round(2 * self.map.scale)):\n            self.use_stream(S_ZOMBIE, self.zombie_spawn_index); self.zombie_spawn_index += 1\n            self.spawn_zombie_at_infected()\n\n    def tick(self, dt):\n        if self.status != "running":\n            return\n        self.time += dt\n        self._roads(); self._infection(dt); self._tasks(dt); self._batches(); self._zombie_spawns(dt)\n        if self.cfg["escape"]:\n            self._check_medevac(dt)\n            self._update_generators()\n            self._spawn_incidents(dt)\n            self._roadblocks(dt)\n        else:\n            self._events(dt)\n        self._restock(dt); self._bots(dt)\n        for p in self.players:\n            self._update_player(p, dt)\n        if self.cfg["escape"]:\n            self._update_incidents(dt)\n        self._noises(dt)\n        if self.time >= self.zombie_stun_until:\n            for z in self.zombies:\n                self._update_zombie(z, dt)\n        if self.cfg["escape"] and self.time >= self.guard_down_until:\n            ex = self.extraction\n            n0 = len(self.zombies)\n            self.zombies = [z for z in self.zombies if not (z.node == ex and z.next_node < 0)]\n            if len(self.zombies) < n0:\n                self.guard_kills += n0 - len(self.zombies)\n\n            self.guard_cooldown -= dt\n            if self.guard_cooldown <= 0:\n                expos, best, bd = self.map.nodes[ex].pos, None, self.cfg["guardRange"]\n                for z in self.zombies:\n                    d = _vd(z.pos, expos)\n                    if d <= bd:\n                        best, bd = z, d\n                if best is not None:\n                    self.zombies.remove(best)\n                    self.guard_kills += 1\n                    self.guard_cooldown = self.cfg["guardFireInterval"]\n                    self.guard_shots = (self.guard_shots + [(self.time, best.pos)])[-20:]\n        self._weapons(dt)\n        self._damage(dt)\n        if self.cfg["escape"]:\n            self._helicopter()\n        elif self.city_infection >= 100:\n            self.end("城市感染程度達到上限，提前淪陷")\n        elif self.time >= self.cfg["matchDuration"]:\n            self.end("時間到，城市全面淪陷")\n        elif self.players and all(p.state == "eliminated" for p in self.players):\n            self.end("所有傭兵皆已淘汰")\n\n    def _helicopter(self):\n        c = self.cfg\n        if self.ready_at < 0 and self.parts >= self.parts_required:\n            self.ready_at = self.time\n            for p in self.players:\n                if p.state == "active":\n                    p.hp = 100.0\n                    p.infection = min(p.infection, 30.0)\n            before = len(self.zombies)\n            self.zombies = [z for z in self.zombies if not self.in_zone(z.pos)]\n            self._spawn_fuel()\n            self.log("heli", f"直升機修好了！守軍清掉圈內 {before - len(self.zombies)} 隻殭屍，去撿回 {self.fuel_required} 桶油。現在倒地就無法復活！", self.extraction)\n        if self.ready_at >= 0 and self.depart_at < 0 and self.fuel >= self.fuel_required:\n            self.depart_at = self.time + c["boardTime"]\n            self.log("heli", f"油加滿了！倒數暫停，{c[\'boardTime\']:.0f} 秒後起飛，快回撤離點！", self.extraction)\n        if self.depart_at >= 0 and self.time >= self.depart_at:\n            self.end("直升機起飛")\n        elif self.depart_at < 0 and self.time >= c["matchDuration"]:\n            self.end("時間到，直升機沒修好" if self.ready_at < 0 else "時間到，油不夠，直升機飛不起來")\n        elif self.players and all(p.state == "eliminated" for p in self.players):\n            self.end("所有人都倒下了")\n\n    @property\n    def no_respawn(self):\n        return self.cfg["escape"] and self.ready_at >= 0\n\n    def doomed(self, p):\n        return self.cfg["escape"] and not self.in_zone(p.pos)\n\n    def _spawn_fuel(self):\n        c, m = self.cfg, self.map\n        self.use_stream(S_EVENT, 100000)\n        n = max(1, len(self.players))\n        self.fuel_required = max(2, _round(n * c["fuelPerPlayer"]) - self.fuel_discount)\n\n        def ring(nd):\n            r, cc = nd.id // m.cols, nd.id % m.cols\n            return min(r, m.rows - 1 - r, cc, m.cols - 1 - cc)\n        pool = [nd for nd in m.nodes if nd.type == "road" and m.dist[self.extraction, nd.id] >= 6 and self.in_zone(nd.pos)]\n        if not pool:\n            pool = [nd for nd in m.nodes if nd.type == "road" and m.dist[self.extraction, nd.id] >= 6] or \\\n                   [nd for nd in m.nodes if nd.type == "road"]\n        edge = min(ring(nd) for nd in pool)\n        far = [nd for nd in pool if ring(nd) <= edge]\n        if len(far) < 4:\n            far = [nd for nd in pool if ring(nd) <= edge + 1]\n        reward = _round(c["fuelReward"] * (1 + 0.25 * self.phase))\n        for _ in range(self.fuel_required + c["fuelSpare"] + int(n * c["fuelSparePerPlayer"])):\n            node = far[self.rng.next(len(far))].id\n            self.tasks.append(Task(id=self._id("task"), type="fuel", origin=node, dest=self.extraction, weight=int(c["fuelWeight"]),\n                                   reward=reward, available_at=self.time, expires_at=-1.0, status="open"))\n\n    @property\n    def zone_radius(self):\n        c = self.cfg\n        t = self.time if self.ready_at < 0 else min(self.time, self.ready_at)\n        z0, z1 = c["zoneStart"] * self.dscale, c["zoneEnd"] * self.dscale\n        if not c["escape"] or t <= z0:\n            return self.zone_max\n        f = min(1.0, (t - z0) / max(1.0, z1 - z0))\n        return self.zone_max + (c["zoneMinRadius"] - self.zone_max) * f\n\n    def is_safe(self, node):\n        n = self.map.nodes[node]\n        if self.cfg["escape"] and node == self.extraction and self.time < self.guard_down_until:\n            return False\n        return n.safe\n\n    def in_zone(self, pos):\n        ex = self.map.nodes[self.extraction].pos\n        return math.hypot(pos[0] - ex[0], pos[1] - ex[1]) <= self.zone_radius\n\n    def _roads(self):\n        changed = False\n        for e in self.map.edges:\n            if e.blocked and self.time >= e.blocked_until:\n                e.blocked = False\n                changed = True\n        if changed:\n            self.map.recompute()\n\n    def _infection(self, dt):\n        m = self.map\n        avg = sum(n.infection for n in m.nodes) / len(m.nodes)\n        c = self.cfg\n        rate = c["coopCityRate"] + c["cityPerPlayer"] * len(self.players) if c["coop"] and not c["escape"] else c["cityRate"]\n        self.city_infection += (100 / self.cfg["matchDuration"]) * rate * (0.6 + avg) * dt\n        if c["escape"]:\n            self.city_infection = min(100.0, self.city_infection)\n        self.spread_timer -= dt\n        if self.spread_timer > 0:\n            return\n        self.spread_timer = 4.0\n        self.use_stream(S_SPREAD, self.spread_tick); self.spread_tick += 1\n        chance = 0.10 + 0.35 * self.city_infection / 100\n        add = [0.0] * len(m.nodes)\n        for n in m.nodes:\n            if n.infection < F06:\n                continue\n            for ei in n.edges:\n                e = m.edges[ei]\n                if not e.blocked and self.rng.next_double() < chance:\n                    add[e.other(n.id)] += self.cfg["spreadAmount"]\n        f = np.float32\n        for n in m.nodes:\n            cap = f(0.4) if n.safe else f(1.0)\n            n.infection = float(min(cap, f(f(n.infection) + f(add[n.id])) + (f(self.cfg["spreadGrow"]) if n.infection > 0 else f(0))))\n\n    def _tasks(self, dt):\n        for t in self.tasks:\n            if t.status == "announced" and self.time >= t.available_at:\n                t.status = "open"\n            if t.status == "open" and t.expires_at > 0 and self.time >= t.expires_at:\n                t.status = "expired"\n                if self.cfg["escape"]:\n                    if t.type == "rescue":\n                        for _ in range(self.cfg["civilianZombies"]):\n                            self.spawn_zombie(t.origin, -1)\n                        self.log("fail", f"倖存者 #{t.id} 沒人救，變成了殭屍", t.origin)\n                elif self.cfg["coop"]:\n                    if t.urgent:\n                        self.city_infection += self.cfg["urgentFailCity"]\n                        self.log("fail", f"緊急委託 #{t.id} 沒人處理，城市感染 +{self.cfg[\'urgentFailCity\']:.0f}%", t.origin)\n                    elif t.type == "rescue":\n                        self.city_infection += self.cfg["rescueFailCity"]\n                elif t.type == "rescue":\n                    self.city_infection += 1\n        self.tasks = [t for t in self.tasks if not (t.status in ("done", "expired") and self.time - t.expires_at > 30 and t.carrier < 0)]\n        self.task_timer -= dt\n        if self.task_timer > 0 or (self.cfg["escape"] and self.ready_at >= 0):\n            return\n        self.task_timer = self.cfg["taskInterval"] / (self.map.scale * self.task_scale * (1 + 0.35 * self.phase))\n        self.use_stream(S_TASK, self.task_slot); self.task_slot += 1\n        self.spawn_task(self.rng.next_double() < 0.15 + 0.07 * self.phase, self.cfg["announceLead"])\n\n    def spawn_task(self, urgent, lead):\n        m, rng, c, ph = self.map, self.rng, self.cfg, self.phase\n        available_at = self.time + lead\n        rescue = not c["escape"] and rng.next_double() < 0.25 + 0.1 * ph\n        if rescue:\n            pool = [n for n in m.nodes if n.type == "road" and n.infection >= F03] or [n for n in m.nodes if n.type == "road"]\n            origin = pool[rng.next(len(pool))].id\n            if c["escape"]:\n                dest = self.extraction\n            else:\n                safe = [n for n in m.nodes if n.safe]\n                dest = safe[rng.next(len(safe))].id\n            weight = 4\n        else:\n\n            src = [n for n in m.nodes if n.type == "factory" and self.powered(n)] if c["escape"] else \\\n                  [n for n in m.nodes if n.type in ("factory", "supply")]\n            if not src:\n                return None\n            origin = src[rng.next(len(src))].id\n            if c["escape"]:\n                dest, weight = self.extraction, 1 + rng.next(3)\n            elif not urgent and rng.next_double() < c["batchChance"]:\n                first = self._spawn_batch(origin, available_at)\n                if first is not None:\n                    return first\n            if not c["escape"]:\n                dst = [n for n in m.nodes if n.id != origin and n.type != "factory" and m.dist[origin, n.id] >= 8]\n                if not dst:\n                    return None\n                dest = dst[rng.next(len(dst))].id\n                weight = 1 + rng.next(4)\n        d = float(m.dist[origin, dest])\n        if d >= INF:\n            return None\n        if rescue:\n            base = (18 + d * 0.5 + 20 * m.nodes[origin].infection) * c["rescueRewardMul"]\n        elif c["escape"]:\n            base = 10 + d * 0.6 + weight * c["partWeightReward"]\n        else:\n            base = 8 + d * 0.5 + weight * 2\n        base *= 1 + 0.25 * ph\n        kind = "rescue" if rescue else ("part" if c["escape"] else "delivery")\n        t = Task(id=self._id("task"), type=kind, origin=origin, dest=dest, weight=weight,\n                 reward=_round(base), available_at=available_at, expires_at=available_at + 90,\n                 status="announced" if lead > 0 else "open")\n        if urgent:\n            t.urgent = True\n            t.bonus = _round(base * 0.6)\n            t.deadline = available_at + (d + 15) / c["playerSpeed"] * 1.5 + 25\n        self.tasks.append(t)\n        return t\n\n    def _spawn_batch(self, origin, available_at):\n        m, rng, c = self.map, self.rng, self.cfg\n        centers = [n for n in m.nodes if n.id != origin and 12 <= m.dist[origin, n.id] < INF]\n        if not centers:\n            return None\n        center = centers[rng.next(len(centers))]\n        area = _order_by_random([n for n in m.nodes if n.id != origin and n.type != "factory" and m.dist[center.id, n.id] <= 9], rng)\n        k = min(len(area), c["batchMin"] + rng.next(c["batchMax"] - c["batchMin"] + 1))\n        if k < 2:\n            return None\n        dests = [n.id for n in area[:k]]\n        tour, at, left = 0.0, origin, list(dests)\n        while left:\n            nxt = min(left, key=lambda n: m.dist[at, n])\n            tour += float(m.dist[at, nxt]); at = nxt; left.remove(nxt)\n        b = Batch(self._id("batch"), origin, available_at + 25 + 1.4 * tour / c["playerSpeed"] + 2 * k)\n        first, total = None, 0\n        for dest in dests:\n            w = 1 + rng.next(2)\n            d = float(m.dist[origin, dest])\n            t = Task(id=self._id("task"), type="delivery", origin=origin, dest=dest, weight=w,\n                     reward=_round((6 + d * 0.45 + w * 2) * (1 + 0.25 * self.phase)),\n                     available_at=available_at, expires_at=available_at + 120,\n                     status="announced" if available_at > self.time else "open", batch=b.id, deadline=b.deadline)\n            self.tasks.append(t)\n            b.tasks.append(t.id)\n            total += t.reward\n            first = first or t\n        b.bonus = _round(total * c["batchBonusRatio"])\n        self.batches[b.id] = b\n        return first\n\n    def batch_of(self, t):\n        return self.batches.get(t.batch) if t is not None and t.batch >= 0 else None\n\n    def can_take(self, p, t):\n        if self.cfg["coop"]:\n            return True\n        b = self.batch_of(t)\n        return b is None or b.owner < 0 or b.owner == p.team\n\n    def _batches(self):\n        for b in self.batches.values():\n            if not b.paid and not b.failed and self.time > b.deadline:\n                b.failed = True\n\n    def _zombie_spawns(self, dt):\n        self.zombie_timer -= dt\n        if self.zombie_timer > 0:\n            return\n        self.zombie_timer = (6 - self.phase) / self.map.scale / self.cfg["zombieSpawnMul"]\n        self.use_stream(S_ZOMBIE, self.zombie_spawn_index); self.zombie_spawn_index += 1\n        cap = _round((self.cfg["zombieCapBase"] + self.cfg["zombieCapGrowth"] * self.time / self.cfg["matchDuration"]) * self.map.scale)\n        if sum(1 for z in self.zombies if z.from_player < 0) < cap:\n            self.spawn_zombie_at_infected()\n\n    def spawn_zombie_at_infected(self):\n        pool = [n for n in self.map.nodes if n.infection >= F05 and not n.safe]\n        if pool:\n            self.spawn_zombie(pool[self.rng.next(len(pool))].id, -1)\n\n    def spawn_zombie(self, node, from_player):\n        z = Zombie(self._id("zombie"), node, self.map.nodes[node].pos, from_player, DotNetRandom(self.rng.next()))\n        self.zombies.append(z)\n        return z\n\n    def _events(self, dt):\n        m = self.map\n        self.event_timer -= dt\n        if self.event_timer > 0:\n            return\n        self.event_timer = 25 - 3 * self.phase\n        self.use_stream(S_EVENT, self.event_index); self.event_index += 1\n        roll, ph = self.rng.next_double(), self.phase\n        if roll < 0.3:\n            self.spawn_task(True, 0.0); return\n        if roll < 0.5:\n            sup = [n for n in m.nodes if n.type == "supply"]\n            s = sup[self.rng.next(len(sup))]\n            s.medkits = min(8, s.medkits + 3)\n            self.log("supply", f"物資空投：補給站 {s.id} 新增醫療包（剩 {s.medkits}）")\n            return\n        if ph == 0:\n            self.spawn_task(True, 0.0); return\n        if roll < 0.75:\n            pool = [n for n in m.nodes if n.type == "road" and n.id != m.start_node]\n            n = pool[self.rng.next(len(pool))]\n            n.infection = 1.0\n            for _ in range(2 + ph):\n                self.spawn_zombie(n.id, -1)\n            self.city_infection += 2\n            self.log("outbreak", f"感染爆發！出現 {2 + ph} 隻殭屍", n.id)\n            return\n        opn = [e for e in m.edges if not e.blocked]\n        brs = [e for e in opn if e.bridge]\n        e = brs[self.rng.next(len(brs))] if brs and self.rng.next_double() < 0.5 else opn[self.rng.next(len(opn))]\n        e.blocked = True\n        m.recompute()\n        if not m.connected():\n            e.blocked = False\n            m.recompute()\n            return\n        e.blocked_until = self.time + 45 + self.rng.next_double() * 45\n        self.log("failure", f"有一段{\'橋\' if e.bridge else \'道路\'}壞了，{e.blocked_until - self.time:.0f} 秒後修好")\n\n    def _roadblocks(self, dt):\n        self.roadblock_timer -= dt\n        if self.roadblock_timer > 0:\n            return\n        c, m = self.cfg, self.map\n        self.roadblock_timer = c["roadblockInterval"] / (1 + 0.15 * self.phase)\n        self.use_stream(S_BLOCK, self.roadblock_index)\n        self.roadblock_index += 1\n        opn = [e for e in m.edges if not e.blocked and not e.bridge]\n        if not opn:\n            return\n        for _ in range(5):\n            e = opn[self.rng.next(len(opn))]\n            e.blocked = True\n            m.recompute()\n            if not m.connected():\n                e.blocked = False\n                m.recompute()\n                continue\n            e.blocked_until = self.time + c["roadblockTime"] * (0.75 + 0.5 * self.rng.next_double())\n            self.log("roadblock", f"屍潮堵住了路口 {e.a} 和 {e.b} 之間的路，{e.blocked_until - self.time:.0f} 秒後才散開", e.a)\n            return\n\n    def incident_by_id(self, i):\n        for inc in self.incidents:\n            if inc.id == i:\n                return inc\n        return None\n\n    def _spawn_incidents(self, dt):\n        c, m = self.cfg, self.map\n        self.incident_timer -= dt\n        if self.incident_timer > 0 or self.ready_at >= 0:\n            return\n        n = max(1, len(self.players))\n        self.incident_timer = c["incidentInterval"] * self.dscale / (1 + 0.2 * self.phase) / max(1.0, (n / 6.0) ** 0.5)\n        self.use_stream(S_EVENT, self.event_index); self.event_index += 1\n        if sum(1 for i in self.incidents if i.status == "active" and i.kind != "generator") >= 1 + n // 4:\n            return\n        ph = self.phase\n        kinds = [("outbreak", 0.3), ("airdrop", 0.25), ("trapped", 0.2), ("bridge", 0.15), ("chest", 0.15)]\n        if ph >= 1:\n            kinds += [("guard", 0.15), ("napalm", 0.15), ("decoy", 0.12)]\n        if ph >= 2:\n            kinds += [("heli", 0.15), ("boss", 0.12)]\n        themes = sorted({nd.district for nd in m.nodes}, key=DISTRICTS.index)\n        kinds += [(DISTRICT_EVENT[d], 0.24 / len(themes)) for d in themes]\n        kinds.append(("generator", c["generatorWeight"]))\n        roll = self.rng.next_double() * sum(w for _, w in kinds)\n        kind = kinds[-1][0]\n        for k, w in kinds:\n            if roll < w:\n                kind = k\n                break\n            roll -= w\n        roads = [nd for nd in m.nodes if nd.type == "road" and nd.id != self.extraction]\n        theme = next((d for d, k in DISTRICT_EVENT.items() if k == kind), None)\n        if theme is not None:\n            pool = [nd for nd in roads if nd.district == theme] or roads\n            node = pool[self.rng.next(len(pool))].id\n        elif kind == "generator":\n            gens = self.generators()\n            if not gens:\n                return\n            g = gens[self.rng.next(len(gens))]\n            if not g.powered or self.time - self.last_generator_break < c["generatorCooldown"]:\n                return\n            self._break_generator(g)\n            self.log("incident", f"突發事件：{INCIDENTS[\'generator\'][0]}! 工廠停工了!", g.id)\n            return\n        elif kind in ("guard", "heli"):\n            node = self.extraction\n        elif kind == "trapped":\n            pool = [nd for nd in roads if nd.infection >= F03] or roads\n            node = pool[self.rng.next(len(pool))].id\n        elif kind == "bridge":\n            opn = [e for e in m.edges if not e.blocked]\n            brs = [e for e in opn if e.bridge] or opn\n            e = brs[self.rng.next(len(brs))]\n            e.blocked = True\n            m.recompute()\n            if not m.connected():\n                e.blocked = False\n                m.recompute()\n                return\n            node = e.a\n        else:\n            node = roads[self.rng.next(len(roads))].id\n        inc = Incident(self._id("incident"), kind, node, self.time)\n        if kind == "bridge":\n            inc.edge = e.id\n            e.blocked_until = inc.deadline + 120.0\n        if kind == "outbreak":\n            m.nodes[node].infection = 1.0\n            for _ in range(2 + ph):\n                self.spawn_zombie(node, -1)\n        if kind == "boss":\n            for _ in range(2):\n                self.spawn_zombie(node, -1)\n        self.incidents.append(inc)\n        self.log("incident", f"突發事件：{inc.name}!", node)\n\n    def generators(self):\n        return [n for n in self.map.nodes if n.type == "generator"]\n\n    def _west(self, node):\n        return self.map.west(node)\n\n    def _generators_for(self, factory):\n        gens = self.generators()\n        same = [g for g in gens if self._west(g.id) == self._west(factory.id)]\n        return same or gens\n\n    def powered(self, n):\n        if n.type == "generator":\n            return n.powered\n        return n.type == "factory" and (not self.cfg["escape"] or any(g.powered for g in self._generators_for(n)))\n\n    def _break_generator(self, g):\n        g.powered = False\n        self.last_generator_break = self.time\n        self.incidents.append(Incident(self._id("incident"), "generator", g.id, self.time))\n\n    def _update_generators(self):\n        ph = self.phase\n        if ph == self.last_phase:\n            return\n        self.last_phase = ph\n        if ph < 1 or ph > 2 or self.ready_at >= 0:\n            return\n        gens = self.generators()\n        if not gens:\n            return\n        g = gens[(ph - 1) % len(gens)]\n        if not g.powered:\n            return\n        self._break_generator(g)\n        self.log("incident", f"突發事件：{INCIDENTS[\'generator\'][0]}! 工廠停工了!", g.id)\n\n    def _spawn_civilians(self):\n        m = self.map\n        self.use_stream(S_TASK, 300000)\n        pool = [n for n in m.nodes if n.type == "road" and n.id != self.extraction and 4 <= m.dist[self.extraction, n.id] < INF]\n        if not pool:\n            return\n        count = min(60, max(6, _round(len(self.players) * self.cfg["civiliansPerPlayer"])))\n        for _ in range(count):\n            self._add_civilian(pool[self.rng.next(len(pool))].id)\n\n    def _add_civilian(self, origin):\n        m, c = self.map, self.cfg\n        d = float(m.dist[origin, self.extraction])\n        if d >= INF:\n            return\n        base = (18 + d * 0.5 + 20 * m.nodes[origin].infection) * c["rescueRewardMul"] * (1 + 0.25 * self.phase)\n        self.tasks.append(Task(id=self._id("task"), type="rescue", origin=origin, dest=self.extraction, weight=4,\n                               reward=_round(base), available_at=self.time, expires_at=-1.0, status="open"))\n\n    def _restart_factories(self, g):\n        c, m, k = self.cfg, self.map, 0\n        for f in m.nodes:\n            if f.type != "factory" or g not in self._generators_for(f):\n                continue\n            d = float(m.dist[f.id, self.extraction])\n            if d >= INF:\n                continue\n            weight = 3\n            base = (10 + d * 0.6 + weight * c["partWeightReward"]) * (1 + 0.25 * self.phase)\n            self.tasks.append(Task(id=self._id("task"), type="part", origin=f.id, dest=self.extraction, weight=weight,\n                                   reward=_round(base), available_at=self.time, expires_at=self.time + 90, status="open"))\n            k += 1\n        return k\n\n    def _check_medevac(self, dt):\n        self.medevac_timer -= dt\n        if self.medevac_timer > 0:\n            return\n        self.medevac_timer = 5.0\n        n = len(self.players)\n        down = sum(1 for p in self.players if p.state in ("downed", "eliminated"))\n        if down < max(2, -(-n // 4)) or self.time < self.medevac_next or \\\n                any(i.kind == "medevac" and i.status == "active" for i in self.incidents):\n            return\n        self.use_stream(S_EVENT, 200000 + self.medevac_rolls); self.medevac_rolls += 1\n        if self.rng.next_double() >= 0.5:\n            return\n        m = self.map\n        pool = [nd for nd in m.nodes if nd.type == "road" and 4 <= m.dist[self.extraction, nd.id] <= 15 and self.in_zone(nd.pos)] or \\\n               [nd for nd in m.nodes if nd.type == "road" and nd.id != self.extraction]\n        node = pool[self.rng.next(len(pool))].id\n        inc = Incident(self._id("incident"), "medevac", node, self.time)\n        self.incidents.append(inc)\n        self.medevac_next = inc.deadline + 60.0\n        self.log("incident", f"突發事件：{inc.name}! 接應到就讓倒下的人全部復活!", node)\n\n    def _update_incidents(self, dt):\n        for inc in self.incidents:\n            if inc.status != "active":\n                continue\n            workers = [p for p in self.players if p.state == "active" and p.next_node < 0 and p.node == inc.node\n                       and p.pending is not None and p.pending["action"] == "fix" and p.pending["task"] == inc.id]\n            inc.workers = len(workers)\n            if workers:\n                for p in workers:\n                    inc.progress += dt\n                    inc.work[p.id] = inc.work.get(p.id, 0.0) + dt\n            if inc.progress >= inc.need:\n                self._solve(inc, workers)\n            elif self.time >= inc.deadline:\n                self._fail_incident(inc)\n        self.incidents = [i for i in self.incidents if i.status == "active" or self.time - i.deadline < 10]\n\n    def _combo_gain(self, p, base):\n        c = self.cfg\n        prev = p.combo\n        gap = self.time - p.last_score_at\n        if gap > c["comboWindow"]:\n            p.combo = 0\n        elif gap >= 2.0:\n            p.combo = min(c["comboMax"], p.combo + 1)\n        p.last_score_at = self.time\n        if p.combo != prev and p.combo in (2, c["comboMax"]):\n            self.log("combo", f"{p.name} 連續得分 {p.combo + 1} 次！", p.node)\n        return base + base * p.combo * c["comboStep"] // 100\n\n    def _solver_names(self, inc):\n        names = [self.player_by_id(pid).name for pid, _ in sorted(inc.work.items(), key=lambda kv: -kv[1])]\n        if not names:\n            return "大家"\n        if len(names) == 1:\n            return names[0]\n        if len(names) <= 3:\n            return "、".join(names[:-1]) + " 和 " + names[-1]\n        return f"{names[0]}、{names[1]} 等 {len(names)} 人"\n\n    def combo_of(self, p):\n        return p.combo if self.time - p.last_score_at <= self.cfg["comboWindow"] else 0\n\n    def _solve(self, inc, workers):\n        c, m = self.cfg, self.map\n        inc.status = "solved"\n        total = sum(inc.work.values()) or 1.0\n        reward = _round(inc.reward * (1 + 0.25 * self.phase) * c["incidentRewardMul"])\n        for pid, w in inc.work.items():\n            q = self.player_by_id(pid)\n            q.score += self._combo_gain(q, _round(reward * w / total))\n            q.fixed += 1\n        for p in workers:\n            p.pending, p.action_timer = None, 0.0\n        pos = m.nodes[inc.node].pos\n        if inc.kind == "outbreak":\n            self.zombies = [z for z in self.zombies if _vd(z.pos, pos) > 4.0]\n            m.nodes[inc.node].infection = 0.0\n            effect = "當地殭屍清除、感染歸零!"\n        elif inc.kind == "bridge":\n            e = m.edges[inc.edge] if 0 <= inc.edge < len(m.edges) else None\n            if e is not None:\n                e.blocked_until = self.time\n            self.speed_boost_until = self.time + c["speedBoostTime"]\n            effect = f"全員加速 {c[\'speedBoostTime\']:.0f} 秒!"\n        elif inc.kind == "airdrop":\n            for p in self.players:\n                if p.state == "active":\n                    p.hp = 100.0\n            for n in m.nodes:\n                if n.type == "supply":\n                    n.medkits = max(n.medkits, 5)\n            effect = "全員生命回滿、補給站醫療包補滿!"\n        elif inc.kind == "trapped":\n            for _ in range(c["trappedCivilians"]):\n                self._add_civilian(inc.node)\n            effect = f"{c[\'trappedCivilians\']} 位民眾等著被帶回撤離點!"\n        elif inc.kind == "guard":\n            self.zombies = [z for z in self.zombies if _vd(z.pos, pos) > 6.0]\n            effect = "守衛擊退了撤離點附近的殭屍!"\n        elif inc.kind == "medevac":\n            back = [q for q in self.players if q.state in ("downed", "eliminated")]\n            for q in back:\n                self._respawn(q, quiet=True)\n            effect = f"{len(back)} 位倒下的同學全部復活!"\n        elif inc.kind == "napalm":\n            n0 = len(self.zombies)\n            self.zombies = [z for z in self.zombies if _vd(z.pos, pos) > 8.0]\n            effect = f"炸掉了附近 {n0 - len(self.zombies)} 隻殭屍!"\n        elif inc.kind == "decoy":\n            self.zombie_stun_until = self.time + c["stunTime"]\n            effect = f"全城殭屍暫停 {c[\'stunTime\']:.0f} 秒!"\n        elif inc.kind == "chest":\n            extra = max(1, _round(self.parts_required * 0.04))\n            self.parts += extra\n            effect = f"寶箱裡有 {extra} 個零件和 {reward} 分!"\n        elif inc.kind == "herbs":\n            for q in self.players:\n                if q.state == "active":\n                    q.medkits += 1\n                    q.hp = min(100.0, q.hp + 30)\n            effect = "全員多一個醫療包、生命 +30!"\n        elif inc.kind == "gas":\n            if self.ready_at >= 0:\n                self.fuel_required = max(1, self.fuel_required - 1)\n            else:\n                self.fuel_discount += 1\n            effect = "之後少要 1 桶油!"\n        elif inc.kind == "rubble":\n            extra = max(1, _round(self.parts_required * 0.05))\n            self.parts += extra\n            effect = f"挖出 {extra} 個零件!"\n        elif inc.kind == "bell":\n            town = [t for t in self.tasks if t.type == "rescue" and t.status in ("open", "announced")\n                    and m.nodes[t.origin].district == "town"]\n            for t in town:\n                t.status, t.expires_at = "done", self.time\n            self.evacuated += len(town)\n            effect = f"鎮民聽到鐘聲! {len(town)} 名平民逃到避難所"\n        elif inc.kind == "boss":\n            n0 = len(self.zombies)\n            self.zombies = [z for z in self.zombies if _vd(z.pos, pos) > 6.0]\n            effect = f"殭屍王倒下! 附近 {n0 - len(self.zombies)} 隻殭屍一同陣亡!"\n        elif inc.kind == "generator":\n            m.nodes[inc.node].powered = True\n            inc.deadline = self.time\n            effect = f"工廠恢復生產零件!"\n        else:\n            self.heli_boost = True\n            if self.ready_at >= 0:\n                self.fuel_required = max(1, self.fuel_required - 1)\n                effect = "直升機少需要 1 桶油!"\n            else:\n                less = max(1, _round(self.parts_required * 0.05))\n                self.parts_required = max(self.parts + 1, self.parts_required - less)\n                effect = f"零件需求 -{less}!"\n        ok = INCIDENT_TEXT.get(inc.kind, (f"解決了{inc.name}!", ""))[0]\n        self.log("solved", f"{self._solver_names(inc)} {ok} {effect}", inc.node)\n\n    def _fail_incident(self, inc):\n        c, m = self.cfg, self.map\n        inc.status = "failed"\n        if inc.kind == "outbreak":\n            for _ in range(3):\n                self.spawn_zombie(inc.node, -1)\n            effect = "殭屍又多了 3 隻!"\n        elif inc.kind == "bridge":\n            effect = "橋要再斷 2 分鐘!"\n        elif inc.kind == "airdrop":\n            for _ in range(2):\n                self.spawn_zombie(inc.node, -1)\n            effect = "物資被殭屍佔領!"\n        elif inc.kind == "trapped":\n            for _ in range(4):\n                self.spawn_zombie(inc.node, -1)\n            effect = "受困的民眾變成了 4 隻殭屍!"\n        elif inc.kind == "guard":\n            self.guard_down_until = self.time + c["guardDownTime"]\n            effect = f"撤離點守衛倒下! {c[\'guardDownTime\']:.0f} 秒內無法攻擊殭屍!"\n        elif inc.kind == "medevac":\n            effect = "醫療隊撤走了!"\n        elif inc.kind in ("napalm", "decoy"):\n            for _ in range(2):\n                self.spawn_zombie(inc.node, -1)\n            effect = "訊號引來 2 隻殭屍!"\n        elif inc.kind == "chest":\n            effect = "寶箱被殭屍叼走了!"\n        elif inc.kind == "herbs":\n            for _ in range(2):\n                self.spawn_zombie(inc.node, -1)\n            effect = "草叢裡竄出 2 隻殭屍!"\n        elif inc.kind == "gas":\n            effect = "這次沒裝到油!"\n        elif inc.kind == "rubble":\n            for _ in range(2):\n                self.spawn_zombie(inc.node, -1)\n            effect = "瓦礫裡爬出 2 隻殭屍!"\n        elif inc.kind == "bell":\n            for _ in range(3):\n                self.spawn_zombie(inc.node, -1)\n            effect = "鐘聲引來 3 隻殭屍!"\n        elif inc.kind == "boss":\n            for _ in range(5):\n                self.spawn_zombie(inc.node, -1)\n            effect = "殭屍王帶來 5 隻殭屍!"\n        else:\n            if self.ready_at >= 0:\n                self.fuel_required += 1\n                effect = "油料外洩，要多撿 1 桶油!"\n            else:\n                extra = max(1, _round(self.parts_required * 0.1))\n                self.parts_required += extra\n                effect = f"直升機故障! 零件需求 +{extra}!"\n        bad = INCIDENT_TEXT.get(inc.kind, ("", f"{inc.name}沒有處理!"))[1]\n        self.log("incident_fail", f"✕ {bad} {effect}", inc.node)\n\n    def _restock(self, dt):\n        self.restock_timer -= dt\n        if self.restock_timer > 0:\n            return\n        self.restock_timer = 45.0\n        for n in self.map.nodes:\n            if n.type == "supply" and n.medkits < 5:\n                n.medkits += 1\n            elif n.type == "armory" and n.weapons < self.cfg["weaponStock"]:\n                n.weapons += 1\n\n    def _bots(self, dt):\n        self.bot_timer -= dt\n        if self.bot_timer > 0:\n            return\n        self.bot_timer = 0.25\n        for p in self.players:\n            if p.bot is None or p.state != "active":\n                continue\n            if not (p.plan_index >= len(p.plan) and p.pending is None and p.next_node < 0):\n                p.idle_since = -1.0\n                continue\n            if p.idle_since < 0:\n                p.idle_since = self.time\n            if self.time - p.idle_since < self.cfg["botReactionTime"]:\n                continue\n            steps = p.bot.think(self, p)\n            if self.time - p.last_plan_time >= self.cfg["planCooldown"] or self.is_emergency(steps):\n                self.submit(p, steps)\n\n    def task_by_id(self, i):\n        for t in self.tasks:\n            if t.id == i:\n                return t\n        return None\n\n    def player_by_id(self, i):\n        for p in self.players:\n            if p.id == i:\n                return p\n        return None\n\n    def load(self, p):\n        return sum((self.task_by_id(i).weight if self.task_by_id(i) else 0) for i in p.cargo) + p.medkits + \\\n            (self.cfg["weaponWeight"] if p.ammo > 0 else 0)\n\n    def speed(self, p):\n        s = self.cfg["playerSpeed"] * (1 - self.cfg["loadSpeedPenalty"] * self.load(p))\n        if p.hp < 30:\n            s *= 0.8\n        if self.time < self.speed_boost_until:\n            s *= self.cfg["speedBoost"]\n        return max(0.5, s)\n\n    def is_emergency(self, steps):\n        if not steps or not all(s["action"] in EMERGENCY for s in steps):\n            return False\n        if any(s["action"] in ("use_medkit", "heal") or s["action"] in DROPS for s in steps):\n            return True\n        moves = [s for s in steps if s["action"] in ("move", "goto")]\n        return bool(moves) and 0 <= moves[-1]["node"] < len(self.map.nodes) and self.map.nodes[moves[-1]["node"]].safe\n\n    def submit(self, p, steps, append=False, start_node=-1):\n        c = self.cfg\n        if self.status != "running":\n            return "比賽尚未開始或已結束"\n        if p.state != "active":\n            return f"角色目前無法行動（{p.state}）"\n        steps = [_step(**{k: s.get(k, d) for k, d in (("action", ""), ("node", -1), ("task", -1), ("target", -1), ("seconds", 0.0))})\n                 for s in (steps or [])]\n        for s in steps:\n            s["action"] = str(s["action"]).strip().lower()\n            s["node"], s["task"], s["target"], s["seconds"] = int(s["node"]), int(s["task"]), int(s["target"]), float(s["seconds"])\n            if s["action"] not in ACTIONS:\n                return "未知的 action：" + s["action"]\n        emergency = not append and self.is_emergency(steps)\n        if not emergency and self.time - p.last_plan_time < c["planCooldown"]:\n            return f"送出太頻繁，請在 {c[\'planCooldown\'] - (self.time - p.last_plan_time):.1f} 秒後再試（緊急撤退 / 治療指令不受限制）"\n        if append:\n            p.plan = p.plan[min(p.plan_index, len(p.plan)):] + steps\n        else:\n            p.plan = list(steps)\n            here = p.next_node if p.next_node >= 0 else p.node\n            if steps and steps[0]["action"] == "move" and 0 <= start_node < len(self.map.nodes) and start_node != here:\n                p.plan.insert(0, _step("goto", node=start_node))\n        p.plan = p.plan[:c["maxPlanSteps"]]\n        p.plan_index = 0\n        if not emergency:\n            p.last_plan_time = self.time\n        p.last_error = ""\n        return None\n\n    def _advance(self, ent, speed, dt):\n        e = self.map.find_edge(ent.node, ent.next_node)\n        ent.progress += speed * dt / max(0.01, e.length)\n        if ent.progress < 1:\n            ent.pos = _lerp(self.map.nodes[ent.node].pos, self.map.nodes[ent.next_node].pos, ent.progress)\n            return\n        ent.node, ent.next_node, ent.progress = ent.next_node, -1, 0.0\n        ent.pos = self.map.nodes[ent.node].pos\n\n    def _update_player(self, p, dt):\n        if p.state == "eliminated":\n            return\n        if p.state == "downed":\n            p.downed_timer -= dt\n            if p.downed_timer <= 0:\n                if self.doomed(p):\n                    self._eliminate(p, p.eliminated_reason or "沒人來救")\n                else:\n                    self._respawn(p)\n            return\n        self._drop_now(p)\n        if p.next_node >= 0:\n            self._advance(p, self.speed(p), dt)\n            return\n        if p.pending is not None:\n            p.action_timer -= dt\n            if p.action_timer <= 0:\n                s, p.pending = p.pending, None\n                err = None if s["action"] == "fix" else self._validate(p, s)\n                if err:\n                    self._fail(p, s, err, True)\n                else:\n                    self._complete(p, s)\n            return\n        for _ in range(8):\n            if not (p.plan_index < len(p.plan) and p.next_node < 0 and p.pending is None):\n                break\n            self._start_step(p, p.plan[p.plan_index])\n\n    def _start_step(self, p, s):\n        m, a = self.map, s["action"]\n        if a == "move":\n            if s["node"] == p.node:\n                p.plan_index += 1; return\n            if not 0 <= s["node"] < len(m.nodes):\n                self._fail(p, s, f"move：節點 {s[\'node\']} 不存在"); return\n            e = m.find_edge(p.node, s["node"])\n            if e is None:\n                self._fail(p, s, f"move：{p.node} 與 {s[\'node\']} 之間沒有道路"); return\n            if e.blocked:\n                self._fail(p, s, f"move：道路 {p.node}-{s[\'node\']} 已封鎖"); return\n            p.next_node, p.progress = s["node"], 0.0\n            p.plan_index += 1\n            return\n        if a == "goto":\n            if not 0 <= s["node"] < len(m.nodes):\n                self._fail(p, s, f"goto：節點 {s[\'node\']} 不存在"); return\n            if s["node"] == p.node:\n                p.plan_index += 1; return\n            hop = int(m.next[p.node, s["node"]])\n            if hop < 0:\n                self._fail(p, s, f"goto：無法抵達節點 {s[\'node\']}"); return\n            p.next_node, p.progress = hop, 0.0\n            return\n        if a in DROPS:\n            self._do_drop(p, s)\n            p.plan_index += 1\n            return\n        t = self.task_by_id(s["task"]) if s["task"] >= 0 else None\n        if (a == "pickup" and s["task"] in p.cargo) or (a == "dropoff" and t is not None and t.completed_by == p.id):\n            p.plan_index += 1; return\n        if a == "pickup" and t is not None and t.status == "announced" and t.origin == p.node and self.can_take(p, t):\n            return\n        err = self._validate(p, s)\n        if err:\n            self._fail(p, s, err); return\n        p.pending = s\n        p.action_timer = {"heal": 4.0, "revive": 3.0, "use_medkit": 1.5}.get(\n            a, min(30.0, max(0.1, s["seconds"])) if a == "wait" else min(30.0, max(1.0, s["seconds"] or 10.0)) if a == "fix" else 1.0)\n        p.plan_index += 1\n\n    def _drop_now(self, p):\n        while p.plan_index < len(p.plan) and p.plan[p.plan_index]["action"] in DROPS:\n            self._do_drop(p, p.plan[p.plan_index])\n            p.plan_index += 1\n\n    def _do_drop(self, p, s):\n        at = p.next_node if p.next_node >= 0 and p.progress > 0.5 else p.node\n        a = s["action"]\n        if a == "drop_medkit":\n            if p.medkits > 0:\n                p.medkits -= 1\n            else:\n                p.last_error = f"[{self.time:.0f}s] drop_medkit：沒有醫療包（已略過此步）"\n            return\n        if a == "drop_weapon":\n            if p.ammo > 0:\n                p.ammo = 0\n            else:\n                p.last_error = f"[{self.time:.0f}s] drop_weapon：沒有武器（已略過此步）"\n            return\n        t = self.task_by_id(s["task"])\n        if t is None or s["task"] not in p.cargo:\n            p.last_error = f"[{self.time:.0f}s] drop：你沒有攜帶委託 #{s[\'task\']}（已略過此步）"\n            return\n        p.cargo.remove(t.id)\n        t.status, t.carrier, t.origin = "open", -1, at\n        t.expires_at = -1.0 if t.type == "fuel" else self.time + 60\n        for k in range(len(p.plan) - 1, p.plan_index, -1):\n            if p.plan[k]["action"] == "dropoff" and p.plan[k]["task"] == t.id:\n                del p.plan[k]\n\n    def _weapons(self, dt):\n        c = self.cfg\n        for p in self.players:\n            if p.state != "active" or p.ammo <= 0:\n                continue\n            p.fire_cooldown -= dt\n            if p.fire_cooldown > 0:\n                continue\n            best, bd = None, c["weaponRange"]\n            for z in self.zombies:\n                d = _vd(z.pos, p.pos)\n                if d <= bd:\n                    best, bd = z, d\n            if best is None:\n                continue\n            self.zombies.remove(best)\n            p.ammo -= 1\n            p.kills += 1\n            p.score += c["killReward"]\n            p.fire_cooldown = c["weaponInterval"]\n            self.player_shots = (self.player_shots + [(p.pos, best.pos, self.time)])[-30:]\n\n    def _validate(self, p, s):\n        node, a, cap = self.map.nodes[p.node], s["action"], self.cfg["capacity"]\n        if a == "pickup":\n            t = self.task_by_id(s["task"])\n            if t is None or t.status != "open":\n                return f"pickup：委託 #{s[\'task\']} 已不可領取"\n            if t.origin != p.node:\n                return f"pickup：委託 #{s[\'task\']} 在節點 {t.origin}，你在 {p.node}"\n            if not self.can_take(p, t):\n                return f"pickup：批次合約 B{t.batch} 已由別人負責"\n            if self.load(p) + t.weight > cap:\n                return f"pickup：背包容量不足（{self.load(p)}+{t.weight}>{cap}）"\n        elif a == "dropoff":\n            t = self.task_by_id(s["task"])\n            if t is None or s["task"] not in p.cargo:\n                return f"dropoff：你沒有攜帶委託 #{s[\'task\']}"\n            if t.dest != p.node:\n                return f"dropoff：委託 #{s[\'task\']} 目的地是 {t.dest}，你在 {p.node}"\n        elif a == "take_medkit":\n            if node.type != "supply":\n                return "take_medkit：這裡不是補給站"\n            if node.medkits <= 0:\n                return "take_medkit：補給站已無醫療包"\n            if self.load(p) + 1 > cap:\n                return "take_medkit：背包容量不足"\n        elif a == "take_weapon":\n            if node.type != "armory":\n                return "take_weapon：這裡不是軍火庫"\n            if node.weapons <= 0:\n                return "take_weapon：軍火庫沒有武器了"\n            if p.ammo > 0:\n                return "take_weapon：你已經有武器了"\n            if self.load(p) + self.cfg["weaponWeight"] > cap:\n                return "take_weapon：背包容量不足"\n        elif a == "use_medkit":\n            return None if p.medkits > 0 else "use_medkit：沒有醫療包"\n        elif a == "heal":\n            return None if node.type == "hospital" else "heal：這裡不是醫院"\n        elif a == "fix":\n            inc = self.incident_by_id(s["task"])\n            if inc is None or inc.status != "active":\n                return f"fix：事件 #{s[\'task\']} 已經結束"\n            if inc.node != p.node:\n                return f"fix：事件 #{s[\'task\']} 在節點 {inc.node}，你在 {p.node}"\n        elif a == "revive":\n            t = self.player_by_id(s["target"])\n            if t is None or t.state != "downed":\n                return f"revive：玩家 {s[\'target\']} 不是倒地狀態"\n            if t.node != p.node:\n                return f"revive：玩家 {s[\'target\']} 在節點 {t.node}"\n        return None\n\n    def _complete(self, p, s):\n        a, c = s["action"], self.cfg\n        if a == "pickup":\n            t = self.task_by_id(s["task"])\n            t.status, t.carrier = "carried", p.id\n            p.cargo.append(t.id)\n            self._noise(p.node, c["actionNoiseRadius"], c["actionNoiseDuration"], "pickup")\n            b = self.batch_of(t)\n            if b is not None and b.owner < 0 and not c["coop"]:\n                b.owner = p.team\n                for i in b.tasks:\n                    bt = self.task_by_id(i)\n                    if bt:\n                        bt.expires_at = max(bt.expires_at, b.deadline + 60)\n                self.log("batch", f"{p.name} 承接批次合約 B{b.id}")\n        elif a == "dropoff":\n            t = self.task_by_id(s["task"])\n            p.cargo.remove(t.id)\n            if not (c["escape"] and p.node == self.extraction):\n                self._noise(p.node, c["actionNoiseRadius"], c["actionNoiseDuration"], "dropoff")\n            useless = c["escape"] and ((t.type == "part" and self.ready_at >= 0) or (t.type == "fuel" and self.fuel >= self.fuel_required))\n            gain = t.reward\n            on_time = not useless and t.urgent and self.time <= t.deadline\n            if on_time:\n                gain += t.bonus\n            if c["escape"] and t.type == "part" and c["partDecay"] > 0:\n                gain = _round(gain * c["partDecay"] / (c["partDecay"] + p.parts))\n            if useless:\n                gain = 0\n            elif c["escape"]:\n                gain = self._combo_gain(p, gain)\n            p.score += gain\n            t.status, t.carrier, t.completed_by, t.expires_at = "done", -1, p.id, self.time\n            if c["coop"] and not c["escape"]:\n                self._reduce_city(p, gain)\n            if t.type == "part":\n                self.parts += t.weight\n                p.parts += t.weight\n            elif t.type == "fuel":\n                self.fuel += 1\n                p.fuel += 1\n            if t.type == "rescue":\n                p.rescued += 1\n                self.evacuated += 1\n                if not c["coop"]:\n                    self.city_infection = max(0.0, self.city_infection - 1.5)\n            elif t.type not in ("part", "fuel"):\n                p.delivered += 1\n            kind = {"rescue": "救援", "part": "零件", "fuel": "油桶"}.get(t.type, "配送")\n            combo = f"（連續得分 {p.combo + 1} 次）" if c["escape"] and p.combo > 0 else ""\n            self.log("done", f"{p.name} 完成{kind} #{t.id} +{gain}{\'（已經不需要了）\' if useless else \'\'}{\'（限時獎勵）\' if on_time else \'\'}{combo}")\n            b = self.batch_of(t)\n            if b is not None and c["coop"]:\n                b.delivered += 1\n                b.by[p.id] = b.by.get(p.id, 0) + 1\n                if not b.paid and not b.failed and b.delivered >= len(b.tasks):\n                    b.paid = True\n                    for pid, n in b.by.items():\n                        q = self.player_by_id(pid)\n                        share = _round(b.bonus * n / len(b.tasks))\n                        q.score += share\n                        self._reduce_city(q, share)\n                    self.log("batch", f"批次合約 B{b.id} 準時完成！合約獎金 +{b.bonus}（{len(b.by)} 人合力）", p.node)\n            elif b is not None and b.owner == p.team:\n                b.delivered += 1\n                if not b.paid and not b.failed and b.delivered >= len(b.tasks):\n                    b.paid = True\n                    p.score += b.bonus\n                    self.log("batch", f"{p.name} 準時完成批次合約 B{b.id}，合約獎金 +{b.bonus}", p.node)\n        elif a == "take_medkit":\n            self.map.nodes[p.node].medkits -= 1\n            p.medkits += 1\n        elif a == "take_weapon":\n            self.map.nodes[p.node].weapons -= 1\n            p.ammo, p.fire_cooldown = c["weaponShots"], 0.0\n        elif a == "use_medkit":\n            p.medkits -= 1\n            p.hp, p.infection = min(100.0, p.hp + 40), max(0.0, p.infection - 25)\n        elif a == "heal":\n            p.hp, p.infection = 100.0, max(0.0, p.infection - 40)\n        elif a == "revive":\n            t = self.player_by_id(s["target"])\n            t.state, t.hp = "active", 40.0\n            t.eliminated_reason = ""\n            t.protect_until = self.time + 5.0\n            fresh = self.time - t.revived_at >= 30.0\n            t.revived_at = self.time\n            bonus = c["reviveBonus"] if fresh else 0\n            p.score += self._combo_gain(p, bonus) if c["escape"] and fresh else bonus\n            if fresh:\n                p.revives += 1\n            self.log("revive", f"{p.name} 救起了 {t.name}" + (f" +{bonus}" if fresh else ""), p.node)\n\n    def _fail(self, p, s, msg, advanced=False):\n        p.last_error = f"[{self.time:.0f}s] {msg}（已略過此步）"\n        if s["action"] == "move" and not advanced:\n            end = p.plan_index\n            while end + 1 < len(p.plan) and p.plan[end + 1]["action"] == "move":\n                end += 1\n            target = p.plan[end]["node"]\n            del p.plan[p.plan_index:end + 1]\n            if 0 <= target < len(self.map.nodes) and target != p.node and self.map.next[p.node, target] >= 0:\n                p.plan.insert(p.plan_index, _step("goto", node=target))\n            return\n        if not advanced:\n            p.plan_index += 1\n        if s["action"] == "pickup":\n            p.plan[p.plan_index:] = [x for x in p.plan[p.plan_index:] if not (x["action"] == "dropoff" and x["task"] == s["task"])]\n\n    def _down(self, p, reason):\n        c = self.cfg\n        if p.next_node >= 0 and p.progress > 0.5:\n            p.node = p.next_node\n        p.next_node, p.progress = -1, 0.0\n        p.pos = self.map.nodes[p.node].pos\n        p.pending, p.plan, p.plan_index = None, [], 0\n        p.state, p.downed_timer, p.hp = "downed", c["downedTime"], 0.0\n        for i in p.cargo:\n            t = self.task_by_id(i)\n            if t:\n                t.status, t.carrier, t.origin = "open", -1, p.node\n                t.expires_at = -1.0 if t.type == "fuel" else self.time + 60\n        p.cargo = []\n        p.ammo = 0\n        p.downs += 1\n        penalty = c["respawnPenalty"] * (min(p.downs, 3) if c["escape"] else 1)\n        p.score -= penalty\n        p.combo, p.last_score_at = 0, -999.0\n        self.city_infection += 1\n        p.eliminated_reason = reason\n        if self.doomed(p):\n            self.log("down", f"{p.name} 在安全區外倒地了！{c[\'downedTime\']:.0f} 秒內沒人來救就逃不掉了", p.node)\n        else:\n            self.log("down", f"{p.name} 倒地（{reason}），掉落貨物、扣 {penalty} 分，{c[\'downedTime\']:.0f} 秒內有人來救 +{c[\'reviveBonus\']}，不然回撤離點復活", p.node)\n\n    def _respawn(self, p, quiet=False):\n        c = self.cfg\n        p.node, p.next_node, p.progress = (self.extraction if c["escape"] else self.map.start_node), -1, 0.0\n        p.pos = self.map.nodes[p.node].pos\n        p.state, p.hp, p.infection = "active", c["respawnHp"], min(p.infection, 30.0)\n        p.protect_until = self.time + 5.0\n        p.last_plan_time = -999.0\n        p.eliminated_at, p.eliminated_reason = -1.0, ""\n        p.plan, p.plan_index, p.pending = [], 0, None\n        if quiet:\n            return\n        self.log("respawn", f"{p.name} 在{\'撤離點\' if c[\'escape\'] else \'避難所\'}復活", p.node)\n\n    def _eliminate(self, p, reason):\n        if p.next_node >= 0 and p.progress > 0.5:\n            p.node = p.next_node\n        p.next_node = -1\n        p.state, p.eliminated_at, p.eliminated_reason = "eliminated", self.time, reason\n        p.plan, p.pending = [], None\n        for i in p.cargo:\n            t = self.task_by_id(i)\n            if t:\n                t.status, t.carrier, t.origin = "open", -1, p.node\n                t.expires_at = -1.0 if t.type == "fuel" else self.time + 60\n        p.cargo = []\n        for b in self.batches.values():\n            if b.owner == p.team:\n                b.owner, b.failed = -1, True\n        if self.cfg["escape"]:\n            self.log("eliminated", f"{p.name} 倒下了（{reason}），沒能逃出", p.node)\n            return\n        self.spawn_zombie(p.node, p.id)\n        self.city_infection += 4\n        self.log("eliminated", f"{p.name} 倒下了（{reason}），化為殭屍，沒能逃出", p.node)\n\n    def _update_zombie(self, z, dt):\n        if z.next_node >= 0:\n            self._advance(z, self.zombie_speed * (self.cfg["zombieChaseMul"] if z.mode == "chase" else 1.0), dt)\n            return\n        if z.wait > 0:\n            z.wait -= dt\n            if self._prey(z) is None:\n                return\n        self._decide(z)\n        if z.target >= 0 and z.target != z.node:\n            hop = int(self.map.next[z.node, z.target])\n            if hop >= 0:\n                z.next_node, z.progress = hop, 0.0\n            else:\n                z.target = -1\n\n    def _decide(self, z):\n        m = self.map\n        prey = self._prey(z)\n        if prey is not None:\n            z.mode, z.chasing = "chase", prey.id\n            z.target = prey.next_node if prey.next_node >= 0 else prey.node\n            return\n        z.chasing = -1\n        near = [n for n in self.noises if _vd(z.pos, n.pos) <= n.radius]\n        if near:\n            z.mode, z.target = "attracted", min(near, key=lambda n: _vd(z.pos, n.pos)).node\n            return\n        if z.mode != "wander" or z.target < 0 or z.target == z.node:\n            z.mode = "wander"\n            cand = [n for n in m.nodes if not n.safe and 6 <= m.dist[z.node, n.id] <= 18] or \\\n                   [n for n in m.nodes if not n.safe and n.id != z.node]\n            z.target = cand[z.rng.next(len(cand))].id if cand else -1\n\n    def _targets(self, p):\n        out = [s["task"] for s in p.plan[p.plan_index:] if s["action"] == "pickup"]\n        if p.pending and p.pending["action"] == "pickup":\n            out.append(p.pending["task"])\n        return out\n\n    def _fixing(self, p):\n        out = [s["task"] for s in p.plan[p.plan_index:] if s["action"] == "fix"]\n        if p.pending and p.pending["action"] == "fix":\n            out.append(p.pending["task"])\n        return out\n\n    def zombie_path(self, z, max_len=10):\n        path, frm = [], z.node\n        if z.next_node >= 0:\n            path.append(z.next_node)\n            frm = z.next_node\n        if z.target >= 0 and z.target != frm:\n            path += self.map.path(frm, z.target) or []\n        return path[:max_len]\n\n    def _prey(self, z):\n        prey, best = None, self.cfg["zombieSight"]\n        for p in self.players:\n            if p.state != "active" or (p.next_node < 0 and self.is_safe(p.node)):\n                continue\n            d = _vd(z.pos, p.pos)\n            if d < best:\n                best, prey = d, p\n        return prey\n\n    def _noise(self, node, radius, duration, source):\n        for n in self.noises:\n            if n.node == node:\n                n.radius = max(n.radius, radius)\n                if self.time + duration > n.until:\n                    n.until, n.source = self.time + duration, source\n                return\n        self.noises.append(Noise(self._id("noise"), node, self.map.nodes[node].pos, radius, self.time + duration, source))\n\n    def _noises(self, dt):\n        self.noises = [n for n in self.noises if self.time < n.until]\n        for p in self.players:\n            if p.state != "active":\n                continue\n            p.noise_timer -= dt\n            if p.noise_timer <= 0 and any(self.task_by_id(i) and self.task_by_id(i).type == "rescue" for i in p.cargo):\n                p.noise_timer = 1.0\n                self._noise(p.next_node if p.next_node >= 0 and p.progress > 0.5 else p.node, self.cfg["survivorNoiseRadius"], 2.0, "survivor")\n\n    def _damage(self, dt):\n        c = self.cfg\n        scale = 0.7 + 0.15 * self.phase\n        for p in self.players:\n            if p.state != "active" or self.time < p.protect_until:\n                continue\n            here = self.map.nodes[p.node]\n            if c["escape"] and not self.in_zone(p.pos):\n                p.hp -= c["zoneDamage"] * dt\n                p.infection += c["zoneInfection"] * dt\n            if not (p.next_node < 0 and self.is_safe(p.node)) and self.time >= self.zombie_stun_until:\n                hit = False\n                for z in self.zombies:\n                    if _vd(z.pos, p.pos) <= c["zombieAttackRange"]:\n                        p.hp -= c["zombieDamage"] * scale * dt\n                        p.infection += c["zombieInfection"] * scale * dt\n                        hit = True\n                if hit and self.time - p.last_scream > 3:\n                    p.last_scream = self.time\n                    self._noise(p.next_node if p.next_node >= 0 and p.progress > 0.5 else p.node, c["screamRadius"], 6.0, "scream")\n                if p.next_node < 0 and here.infection > F07:\n                    p.infection += 0.5 * dt\n            if p.infection >= 100:\n                p.infection = 100.0\n                self._down(p, "感染值達到上限")\n            elif p.hp <= 0:\n                self._down(p, "被殭屍咬倒")\n\n    def _reduce_city(self, p, points):\n        d = min(self.city_infection, points * self.cfg["cityPerPoint"])\n        self.city_infection -= d\n        p.city_reduced += d\n\n    def end(self, reason):\n        self.status, self.end_reason = "ended", reason\n        if self.cfg["escape"]:\n            self.class_won = self.depart_at >= 0 and self.time >= self.depart_at\n            if self.class_won:\n                for p in self.players:\n                    if p.state == "active" and p.next_node < 0 and p.node == self.extraction:\n                        p.escaped = True\n                        p.score += self.cfg["escapeBonus"] + _round(self.time_left() * self.cfg["earlyBonusPerSec"])\n                n = sum(p.escaped for p in self.players)\n                self.end_reason = f"直升機起飛！{n}/{len(self.players)} 人逃出城市"\n            elif self.ready_at < 0:\n                self.end_reason = f"直升機沒修好（零件 {self.parts}/{self.parts_required}），全班被困在城市裡"\n            elif self.players and all(p.state == "eliminated" for p in self.players):\n                self.end_reason = "所有人都倒下了，直升機沒能起飛"\n            else:\n                self.end_reason = f"油不夠（{self.fuel}/{self.fuel_required} 桶），直升機飛不起來"\n            for i, p in enumerate(self.ranking()):\n                p.rank = i + 1\n            return\n        self.class_won = self.cfg["coop"] and self.city_infection < 100\n        if self.class_won:\n            self.end_reason = "全班撐過了！城市守住了"\n            for p in self.players:\n                p.score += self.cfg["classBonus"]\n        for p in self.players:\n            if not self.cfg["escape"] and p.state == "active" and p.next_node < 0 and self.map.nodes[p.node].safe:\n                p.score += self.cfg["survivalBonus"]\n                p.evacuated = True\n        for i, p in enumerate(self.ranking()):\n            p.rank = i + 1\n\n    TITLES = [("零件王", "parts"), ("救援王", "rescued"), ("事件王", "fixed"), ("油桶王", "fuel"), ("醫護王", "revives")]\n\n    def ranking(self):\n        return sorted(self.players, key=lambda p: (-p.score, p.state != "active",\n                                                   -(1e30 if p.eliminated_at < 0 else p.eliminated_at), p.id))\n\n    def replay_header(self, every):\n        m = self.map\n        return dict(every=every, extraction=self.extraction, duration=self.cfg["matchDuration"],\n                    nodes=[[round(n.pos[0], 2), round(n.pos[1], 2), n.type] for n in m.nodes],\n                    edges=[[e.a, e.b, 1 if e.bridge else 0] for e in m.edges],\n                    players={str(p.id): [p.name, p.color] for p in self.players})\n\n    def replay_frame(self):\n        r2 = lambda v: round(v, 2)\n        st = {"active": 0, "downed": 1, "eliminated": 2}\n        return dict(\n            t=r2(self.time), z=r2(self.zone_radius), parts=self.parts, preq=self.parts_required, fuel=self.fuel,\n            freq=self.fuel_required, ready=r2(self.ready_at), depart=r2(self.depart_at),\n            p=[[p.id, r2(p.pos[0]), r2(p.pos[1]), st.get(p.state, 0), p.score, round(p.hp)] for p in self.players],\n            zb=[[r2(z.pos[0]), r2(z.pos[1]), 1 if z.mode == "chase" else 0] for z in self.zombies],\n            k=[[t.origin, t.type[0], 1 if t.status == "announced" else 0] for t in self.tasks if t.status in ("open", "announced")],\n            i=[[i.node, i.kind] for i in self.incidents if i.status == "active"],\n            b=[e.id for e in self.map.edges if e.blocked],\n            g=[n.id for n in self.map.nodes if n.type == "generator" and n.powered])\n\n    def state_dict(self, viewer: Player | None):\n        c, m = self.cfg, self.map\n\n        def unit(p):\n            return dict(id=p.id, name=p.name, state=p.state, node=p.node, next_node=p.next_node, hp=p.hp,\n                        infection=p.infection, score=p.score, load=self.load(p), capacity=c["capacity"], medkits=p.medkits, ammo=p.ammo,\n                        speed=self.speed(p), cargo=list(p.cargo), plan=[dict(s) for s in p.plan], plan_index=p.plan_index,\n                        busy=p.pending["action"] if p.pending else ("moving" if p.next_node >= 0 else ""),\n                        last_error=p.last_error, next_plan_at=p.last_plan_time + c["planCooldown"])\n        you = {}\n        if viewer is not None:\n            you = unit(viewer)\n        live = ("announced", "open", "carried")\n        return dict(\n            status=self.status, phase=self.phase, coop=c["coop"], class_won=self.class_won,\n            escape=dict(enabled=c["escape"], extraction=self.extraction, parts=self.parts, parts_required=self.parts_required,\n                        ready_at=self.ready_at, depart_at=self.depart_at, zone_x=m.nodes[self.extraction].pos[0],\n                        zone_y=m.nodes[self.extraction].pos[1], zone_radius=self.zone_radius,\n                        zone_damage=c["zoneDamage"], escape_bonus=c["escapeBonus"], evacuated=self.evacuated,\n                        speed_boost_until=self.speed_boost_until, guard_down_until=self.guard_down_until,\n                        zombie_stun_until=self.zombie_stun_until,\n                        heli_boost=self.heli_boost, fuel=self.fuel, fuel_required=self.fuel_required),\n            incidents=[dict(id=i.id, type=i.kind, name=i.name, node=i.node, deadline=i.deadline, progress=i.progress,\n                            need=i.need, min_people=i.min_people, workers=i.workers, status=i.status,\n                            reward=_round(i.reward * (1 + 0.25 * self.phase))) for i in self.incidents], phase_name=PHASE_KEYS[self.phase], end_reason=self.end_reason,\n            seed=c["seed"], start_node=m.start_node, match_id=1, time=self.time, duration=c["matchDuration"],\n            time_left=max(0.0, c["matchDuration"] - (self.depart_at - c["boardTime"] if c["escape"] and self.depart_at >= 0 else self.time)),\n            city_infection=self.city_infection,\n            rules=dict(capacity=c["capacity"], max_plan_steps=c["maxPlanSteps"], survival_bonus=c["survivalBonus"], respawn_penalty=c["respawnPenalty"],\n                       player_speed=c["playerSpeed"], load_speed_penalty=c["loadSpeedPenalty"], plan_cooldown=c["planCooldown"],\n                       downed_time=c["downedTime"], zombie_speed=c["zombieSpeed"], zombie_speed_per_phase=c["zombieSpeedPerPhase"],\n                       zombie_sight=c["zombieSight"], zombie_attack_range=c["zombieAttackRange"], zombie_damage=c["zombieDamage"],\n                       zombie_infection=c["zombieInfection"], zombie_chase_mul=c["zombieChaseMul"],\n                       action_noise_radius=c["actionNoiseRadius"],\n                       action_noise_duration=c["actionNoiseDuration"], survivor_noise_radius=c["survivorNoiseRadius"],\n                       scream_radius=c["screamRadius"], weapon_weight=c["weaponWeight"], weapon_shots=c["weaponShots"],\n                       kill_reward=c["killReward"], fuel_weight=c["fuelWeight"], weapon_range=c["weaponRange"], weapon_interval=c["weaponInterval"]),\n            nodes=[dict(id=n.id, type=n.type, x=n.pos[0], y=n.pos[1], infection=n.infection, medkits=n.medkits, weapons=n.weapons, safe=n.safe,\n                        powered=self.powered(n), district=n.district) for n in m.nodes],\n            edges=[dict(id=e.id, a=e.a, b=e.b, length=e.length, blocked=e.blocked, bridge=e.bridge,\n                        blocked_until=e.blocked_until if e.blocked else 0) for e in m.edges],\n            tasks=[dict(id=t.id, type=t.type, origin=t.origin, dest=t.dest, weight=t.weight, reward=t.reward, bonus=t.bonus,\n                        urgent=t.urgent, deadline=t.deadline, expires_at=t.expires_at, status=t.status, carrier=t.carrier,\n                        batch=t.batch, available_at=t.available_at) for t in self.tasks if t.status in live],\n            batches=[dict(id=b.id, origin=b.origin, owner=b.owner, bonus=b.bonus, delivered=b.delivered, deadline=b.deadline,\n                          paid=b.paid, failed=b.failed, tasks=list(b.tasks)) for b in self.batches.values()\n                     if any(self.task_by_id(i) and self.task_by_id(i).status in live for i in b.tasks)],\n            players=[dict(id=p.id, name=p.name, color=p.color, state=p.state, bot=p.bot_kind, node=p.node,\n                          next_node=p.next_node, progress=p.progress, x=p.pos[0], y=p.pos[1], hp=p.hp, infection=p.infection,\n                          downed_timer=p.downed_timer, score=p.score, load=self.load(p), delivered=p.delivered, rescued=p.rescued, downs=p.downs,\n                          city_reduced=p.city_reduced, role=p.role, targets=self._targets(p), parts=p.parts, escaped=p.escaped,\n                          fixing=self._fixing(p), fixed=p.fixed, fuel=p.fuel, combo=self.combo_of(p), ammo=p.ammo, kills=p.kills)\n                     for p in self.players],\n            zombies=[dict(id=z.id, node=z.node, next_node=z.next_node, x=z.pos[0], y=z.pos[1], from_player=z.from_player,\n                          progress=z.progress, mode=z.mode, target=z.target, chasing=z.chasing, path=self.zombie_path(z))\n                     for z in self.zombies],\n            noises=[dict(id=n.id, node=n.node, x=n.pos[0], y=n.pos[1], radius=n.radius, until=n.until, source=n.source) for n in self.noises],\n            zombie_speed=self.zombie_speed,\n            events=[dict(time=e["time"], type=e["type"], text=e["text"]) for e in self.events[-20:]],\n            you=you,\n        )\n\ndef classmate_policy():\n    from apocalypse_sdk import CityGraph, drop, dropoff, fix, goto, heal, pickup, revive, use_medkit\n\n    def policy(st):\n        me = st.you\n        g = CityGraph(st, zombie_penalty=3, zombie_lookahead=10)\n        ex, depart = st.extraction, st.escape.get("depart_at", -1)\n        if depart > 0 and depart - st.time < g.dist(st.here, ex) / me["speed"] + 15:\n            if st.here == ex and st.is_idle:\n                drops = [dropoff(t["id"]) for t in st.my_cargo() if t["dest"] == ex]\n                return drops if drops and st.time >= me["next_plan_at"] else None\n            going = me["plan"] and me["plan"][-1]["node"] == ex\n            return None if going else [goto(ex)]\n        if not st.is_idle:\n            return None\n        if me["hp"] < 40 or me["infection"] > 60:\n            if me["medkits"] > 0:\n                return [use_medkit()]\n            return [goto(g.nearest(st.here, st.nodes_of("hospital"))), heal()]\n        e = st.escape\n        useless = [t for t in st.my_cargo() if (t["type"] == "part" and e["ready_at"] >= 0)\n                   or (t["type"] == "fuel" and e["fuel"] >= e["fuel_required"])]\n        if useless:\n            return [drop(t["id"]) for t in useless]\n        if st.time < me["next_plan_at"]:\n            return None\n        for q in st.players:\n            if q["state"] == "downed" and q["id"] != me["id"]:\n                eta = g.dist(st.here, q["node"]) / me["speed"]\n                others = [g.dist(o["node"], q["node"]) for o in st.players if o["state"] == "active" and o["id"] not in (me["id"], q["id"])]\n                if eta + 4 < q["downed_timer"] and g.dist(st.here, q["node"]) <= min(others, default=1e9):\n                    return [goto(q["node"]), revive(q["id"])]\n        if st.my_cargo():\n            return [s for t in st.my_cargo() for s in (goto(t["dest"]), dropoff(t["id"]))]\n        for inc in sorted(st.active_incidents(), key=lambda i: g.dist(st.here, i["node"])):\n            eta = g.dist(st.here, inc["node"]) / me["speed"]\n            if (eta < 25 or inc["type"] == "generator") and eta + 3 < inc["deadline"] - st.time and st.helpers(inc["id"]) < inc["min_people"]:\n                return [goto(inc["node"]), fix(inc["id"], 20)]\n        claimed = st.claimed_tasks()\n        tasks = [t for t in st.open_tasks() if t["id"] not in claimed]\n        if st.escape.get("ready_at", -1) >= 0:\n            fuel = [t for t in tasks if t["type"] == "fuel"] if st.escape["fuel"] < st.escape["fuel_required"] else []\n            if not fuel:\n                return None if st.here == ex else [goto(ex)]\n            tasks = fuel\n        if not tasks:\n            return None\n        t = min(tasks, key=lambda t: (g.dist(st.here, t["origin"]) + g.dist(t["origin"], t["dest"])) / max(1, t["reward"]))\n        return [goto(t["origin"]), pickup(t["id"]), goto(t["dest"]), dropoff(t["id"])]\n    return policy\n\nclass LocalGame:\n\n    def __init__(self, seed: int = 1, name: str = "我", frame_every: float = 3.0, classmates: int = 11,\n                 classmate_policies=None, role: str = "", **cfg):\n        self.sim = Sim(seed=seed, **cfg)\n        self.me = self.sim.add_player(name)\n        self.me.role = role\n        self.agents = []\n        pols = classmate_policies or [classmate_policy() for _ in range(classmates)]\n        for i, pol in enumerate(pols):\n            self.agents.append((self.sim.add_player(f"同學{i + 1}"), pol))\n        self.frame_every = frame_every\n        self.frames: list = []\n        self.replay_every = 0.5\n        self.replay_frames: list = []\n        self.replay_head = None\n        self.errors: list[str] = []\n\n    def state(self, p=None):\n        from apocalypse_sdk import State\n        return State(self.sim.state_dict(p or self.me))\n\n    def run(self, policy, interval: float = 1.0, latency: float = 0.3, dt: float = 0.05, verbose: bool = False):\n        sim = self.sim\n        sim.start()\n        agents = [(self.me, policy)] + self.agents\n        queues = {p.id: [] for p, _ in agents}\n        import random\n        jitter = random.Random(sim.cfg["seed"])\n\n        offset = {p.id: (0.0 if p is self.me else jitter.random() * interval) for p, _ in agents}\n        lag = {p.id: (latency if p is self.me else 0.05 + 0.35 * jitter.random()) for p, _ in agents}\n        next_call = {p.id: offset[p.id] for p, _ in agents}\n        next_frame = next_replay = 0.0\n        self.replay_head = sim.replay_header(self.replay_every)\n        while sim.status == "running":\n            if sim.time >= next_frame:\n                self.frames.append(self.state())\n                next_frame += self.frame_every\n            if sim.time >= next_replay:\n                self.replay_frames.append(sim.replay_frame())\n                next_replay += self.replay_every\n            for p, pol in agents:\n                if sim.time >= next_call[p.id]:\n                    next_call[p.id] += interval\n                    if p.state != "active":\n                        continue\n                    if p is not self.me and not (p.plan_index >= len(p.plan) and p.pending is None and p.next_node < 0):\n                        continue\n                    st = self.state(p)\n                    t0 = _time.perf_counter()\n                    plans = pol(st)\n                    spent = _time.perf_counter() - t0 if p is self.me else 0.0\n                    if isinstance(plans, dict):\n                        plans = plans.get(0)\n                    if plans is not None:\n                        queues[p.id].append((sim.time + lag[p.id] + spent, plans, st.here))\n            for p, _ in agents:\n                q = queues[p.id]\n                while q and q[0][0] <= sim.time:\n                    _, steps, start = q.pop(0)\n                    err = sim.submit(p, steps, start_node=start)\n                    if err and verbose and p is self.me:\n                        print(f"[{sim.time:5.0f}s] 送出失敗：{err}")\n            if self.me.last_error and (not self.errors or self.errors[-1] != self.me.last_error):\n                if verbose:\n                    print("⚠", self.me.last_error)\n                self.errors.append(self.me.last_error)\n            sim.tick(dt)\n        final = self.state()\n        self.frames.append(final)\n        self.replay_frames.append(sim.replay_frame())\n        return final\n\n    def replay_data(self):\n        return dict(header=self.replay_head, frames=self.replay_frames)\n\n    def result(self) -> dict:\n        sim = self.sim\n        me = self.me\n        mates = [p.score for p, _ in self.agents]\n        return dict(seed=sim.cfg["seed"], score=me.score, rank=me.rank, players=len(sim.players),\n                    alive=me.evacuated, downs=me.downs, delivered=me.delivered, rescued=me.rescued,\n                    end_reason=sim.end_reason, class_won=sim.class_won, ended_at=sim.time, escaped=me.escaped,\n                    parts=me.parts, parts_total=sim.parts, parts_required=sim.parts_required, depart_at=sim.depart_at,\n                    fuel_total=sim.fuel, fuel_required=sim.fuel_required, ready_at=sim.ready_at,\n                    escaped_count=sum(p.escaped for p in sim.players),\n                    scores=sorted((p.score for p in sim.players), reverse=True),\n                    city=sim.city_infection, city_reduced=me.city_reduced,\n                    class_avg=sum(mates) / len(mates) if mates else 0)\n'}
for name, src in FILES.items():
    open(name, "w", encoding="utf-8").write(src)

import matplotlib, matplotlib.font_manager as fm, glob
for f in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    fm.fontManager.addfont(f); matplotlib.rcParams["font.family"] = "Noto Sans CJK JP"

import importlib, apocalypse_sdk, apocalypse_local
importlib.reload(apocalypse_local); importlib.reload(apocalypse_sdk)
from apocalypse_sdk import *
print("安裝完成")

## 遊戲規則

怎麼贏：

1. 把零件搬回撤離點，零件湊滿，直升機就修好了
2. 修好後地上會出現油桶，撿回撤離點把油加滿
3. 油一加滿，時間就停住，45 秒後起飛。起飛時人在撤離點，就算逃出去

時間到了零件還沒湊滿、油還沒加滿，或所有人都被淘汰，全班就輸了。

地圖上有這些地方：

- 撤離點：出發、復活、交東西的地方，殭屍進不來
- 醫院：`heal()` 補滿血
- 補給站：`take_medkit()` 拿醫療包
- 工廠：零件從這裡出來，但同一邊的發電機要有電
- 發電機：河的兩邊各一台，一開場都是壞的，要先去修，不然沒有零件。`st.powered(路口)` 可以查有沒有電
- 軍火庫：一場一個，`take_weapon()` 拿武器（重 4，一次一把）。帶著武器會自動射倒 3.5 格內的殭屍，一把 8 發，每隻 +5 分，倒地時武器會不見。`st.nodes[路口]["weapons"]` 是庫存

要搬的東西全部送回撤離點，背包最多裝 10。每背 1 重量，速度少 5%，背滿只剩一半：

| | 重量 | 說明 |
|---|---|---|
| 零件 | 1–3 | 從有電的工廠拿，放太久沒人拿會消失 |
| 民眾 | 4 | 一開場就決定好，不會變多也不會消失 |
| 油桶 | 4 | 直升機修好後才出現，散在地圖邊緣 |
| 武器 | 4 | 從軍火庫拿 |

直升機修好之後再送零件、油加滿之後再送油桶，都不會加分。用不到的東西可以 `drop(委託id)` 丟掉，會掉在最近的路口。

其他要知道的：

- 1:30 開始安全區會往撤離點縮小，在圈外會一直掉血
- 河每場都不一樣，把地圖切成差不多大的兩半，只有兩座橋，分在河的兩頭
- 路網很疏，很多地方只有一兩條路能到，走哪條路差很多。`CityGraph` 會幫你算最短路
- 屍潮會不定時堵住一段路（不會堵橋），沒辦法處理，只能等或繞路。被堵住的路 `blocked` 是 True，`CityGraph` 會自動避開
- 橋只會因為「橋樑損毀」事件斷掉，修好之前一直斷著
- 殭屍追人時會跑得比平常快，越後期越快、數量也越多。`st.rules` 裡有殭屍的速度和傷害
- 地圖上會出現突發事件，走過去用 `fix(事件id, 秒)` 處理。一個人也能做，人多比較快
- 血歸零或感染滿了會倒地，東西會掉在地上，還會扣分。有人走過來 `revive(同學id)` 就能起來
- 倒地 15 秒沒人扶：在安全區裡會回撤離點復活，在安全區外就淘汰（`st.in_zone(路口)` 可以查）
- 搭上直升機加 200 分，再加上時間停住時剩下的秒數，越早起飛大家加越多

### 寫程式會用到的

`policy(st)` 每秒會被叫一次。回傳一串行動就是新的計畫，回傳 `None` 就照原本的計畫走。

| 行動 | 意思 |
|---|---|
| `goto(路口)` | 走最短路到那個路口 |
| `move(路口)` | 走到隔壁的路口（想自己決定路線時用） |
| `pickup(委託id)` / `dropoff(委託id)` | 撿東西 / 交東西 |
| `take_medkit()` / `use_medkit()` | 拿醫療包 / 用醫療包 |
| `heal()` | 在醫院治療 |
| `wait(秒)` | 原地等 |
| `fix(事件id, 秒)` | 處理突發事件 |
| `take_weapon()` | 在軍火庫拿武器 |
| `drop(委託id)` / `drop_medkit()` / `drop_weapon()` | 丟掉背包裡的東西。只有丟東西的計畫隨時都能送，不用等 15 秒 |

送出計畫後要等 15 秒才能再送（去醫院、回撤離點例外），所以一次要排好一整串。

常用的資料：

- `st.you`：你自己，`st.here`：你在哪個路口，`st.extraction`：撤離點
- `st.open_tasks()`：可以撿的東西，`st.claimed_tasks()`：同學已經要去撿的
- `st.active_incidents()`：進行中的事件，`st.helpers(事件id)`：還有幾個同學要去
- `st.escape["ready_at"]`：直升機什麼時候修好（-1 是還沒），`st.escape["depart_at"]`：什麼時候起飛
- `CityGraph(st, zombie_penalty=3)` 算距離：`g.dist(a, b)`、`g.path(a, b)`、`g.nearest(a, 一群路口)`。`zombie_penalty` 會把附近有殭屍的路算得比較遠，路線就會繞開殭屍，設 0 就不閃

## 分數怎麼算

| 怎麼拿分 | 多少 |
|---|---|
| 交零件 | 直升機修好之後是 0。修好前是 `t["reward"]` × 80 ÷ (80 + 你已經交的零件數)。`reward` 已經算好距離和重量：(10 + 距離×0.6 + 重量×12) × 期間加成 |
| 送民眾回撤離點 | `t["reward"]`：(18 + 距離×0.5 + 感染×20) × 2 |
| 交油桶 | `t["reward"]`：80 × 期間加成。油加滿之後再交就是 0 |
| 限時委託準時送到 | 再加 `t["bonus"]`（報酬的 60%），要在 `t["deadline"]` 前送到 |
| 處理突發事件 | `inc["reward"]` × 2，大家一起做的話照每個人做了幾秒來分 |
| 扶起倒地的同學 | 60 |
| 用武器射倒殭屍 | 每隻 5 |
| 搭上直升機 | 200 + 時間停住時剩下的秒數 |
| 連續得分 | 上一次得分後 25 秒內又得分，每次多 10%，最多 +50%（同一個時間交的貨只算一次） |
| 倒地 | 第 1 次 −50、第 2 次 −100、之後每次 −150 |

期間加成：前期 ×1、中期 ×1.25、後期 ×1.5、終局 ×1.75。委託的 `reward` 在出現的那一刻就算好了。

距離就是 `g.dist(a, b)`。你每秒走 `st.you["speed"]` 格（大約 3，背越重越慢），撿和交東西各要 1 秒。

## 3. 我的策略

這是入門範例，照順序做這幾件事：

1. 直升機快起飛了，就回撤離點
2. 受傷了，先補血
3. 附近有同學倒地，而且我是最近的，就去扶
4. 身上有東西，先送回去
5. 附近有人手不夠的事件，就去幫忙
6. 都沒有的話，挑一件最划算的東西去撿

要改就改這一格。

In [ ]:
#@title 3. 我的策略（改這一格）
def my_make_policy():

    def policy(st):
        me = st.you
        g = CityGraph(st, zombie_penalty=3, zombie_lookahead=10)

        ex, depart = st.extraction, st.escape["depart_at"]
        if depart > 0 and depart - st.time < g.dist(st.here, ex) / me["speed"] + 15:
            if st.here == ex and st.is_idle:
                drops = [dropoff(t["id"]) for t in st.my_cargo() if t["dest"] == ex]
                return drops if drops and st.time >= me["next_plan_at"] else None
            going = me["plan"] and me["plan"][-1]["node"] == ex
            return None if going else [goto(ex)]

        if not st.is_idle:
            return None
        if me["hp"] < 40 or me["infection"] > 60:
            if me["medkits"] > 0:
                return [use_medkit()]
            return [goto(g.nearest(st.here, st.nodes_of("hospital"))), heal()]
        e = st.escape
        useless = [t for t in st.my_cargo() if (t["type"] == "part" and e["ready_at"] >= 0)
                   or (t["type"] == "fuel" and e["fuel"] >= e["fuel_required"])]
        if useless:
            return [drop(t["id"]) for t in useless]
        if st.time < me["next_plan_at"]:
            return None

        for q in st.players:
            if q["state"] == "downed" and q["id"] != me["id"]:
                eta = g.dist(st.here, q["node"]) / me["speed"]
                others = [g.dist(o["node"], q["node"]) for o in st.players
                          if o["state"] == "active" and o["id"] not in (me["id"], q["id"])]
                if eta + 4 < q["downed_timer"] and g.dist(st.here, q["node"]) <= min(others, default=1e9):
                    return [goto(q["node"]), revive(q["id"])]

        cargo = st.my_cargo()
        if cargo:
            return [s for t in cargo for s in (goto(t["dest"]), dropoff(t["id"]))]

        for inc in sorted(st.active_incidents(), key=lambda i: g.dist(st.here, i["node"])):
            eta = g.dist(st.here, inc["node"]) / me["speed"]
            if (eta < 25 or inc["type"] == "generator") and eta + 3 < inc["deadline"] - st.time and st.helpers(inc["id"]) < inc["min_people"]:
                return [goto(inc["node"]), fix(inc["id"], 20)]

        claimed = st.claimed_tasks()
        tasks = [t for t in st.open_tasks() if t["id"] not in claimed]

        if st.escape.get("ready_at", -1) >= 0:
            fuel = [t for t in tasks if t["type"] == "fuel"] if st.escape["fuel"] < st.escape["fuel_required"] else []
            if not fuel:
                return None if st.here == ex else [goto(ex)]
            tasks = fuel
        if not tasks:
            return None

        def cost(t):
            return (g.dist(st.here, t["origin"]) + g.dist(t["origin"], t["dest"])) / max(1, t["reward"])

        t = min(tasks, key=cost)
        return [goto(t["origin"]), pickup(t["id"]), goto(t["dest"]), dropoff(t["id"])]

    return policy

## 4. 本地測試

你加上 11 個 AI 同學，在 Colab 裡把三張地圖各打一場，幾秒鐘就跑完，最後會播重播。

In [ ]:
#@title 4. 本地測試
games = test_local(my_make_policy, seeds=[1, 2, 3], name=MY_NAME)

重播裡有白框、標名字的是你，方塊是設施，小方塊是可以撿的東西，橘色圈是突發事件，綠點是殭屍（變紅代表在追人），紅色虛線圈是安全區。
下面的拉桿可以拖到任何時間點，右邊可以選播放速度，每一格是 0.5 秒。

## 4-2 練習房

自己選地圖和 AI 同學的人數，想打幾場就打幾場。`CLASSMATES` 設 0 就是只有你一個人。

In [ ]:
#@title 4-2 練習房
SEED = 1           #@param {type:"integer"}
CLASSMATES = 11    #@param {type:"slider", min:0, max:29, step:1}
games = test_local(my_make_policy, seeds=[SEED], classmates=CLASSMATES, name=MY_NAME)

## 5. 上場

比賽時執行。想改程式就先按停止，改好再執行，會接回同一個角色。

In [ ]:
#@title 5. 上場
play(my_make_policy, SERVER_URL, MY_NAME)

## 5-2 看剛剛那場的重播

比賽結束後執行，下一場開始前都看得到。拉桿可以拖到任何時間點，右邊可以選播放速度，每一格是 0.5 秒。

In [ ]:
#@title 5-2 看剛剛那場的重播
frames = replay_match(SERVER_URL, MY_NAME)

## 想變強？

挑一個方向，把這句話連同「3. 我的策略」的程式貼給 AI，改完用「4. 本地測試」比比看：

- 一次撿很多：「背包可以裝 10，一趟撿好幾件再一起送回撤離點，用 OR-Tools 決定撿哪些、走什麼順序」
- 事件要不要去：「先算來不來得及、還有幾個同學要去，再決定要不要去」
- 避開圈外：「不要撿安全區外面的東西（`st.in_zone(路口)`），除非分數很高」
- 更怕殭屍：「把 `zombie_penalty` 調大，例如 `CityGraph(st, zombie_penalty=8, zombie_lookahead=15)`」
- 算好回家時間：「直升機起飛前，來不及回撤離點的就不要撿」
- 帶醫療包：「沒有醫療包時，補給站順路的話先拿一個」
- 帶武器：「軍火庫順路的話先拿一把武器（`take_weapon()`），但武器重 4，要算進背包」

## 6. OR-Tools 範例：用 CP-SAT 規劃一趟

保命的事跟入門範例一樣，差別在撿東西：入門範例一次只撿一件，這裡交給 CP-SAT，一趟撿好幾件、排好順序，最後一起送回撤離點。

跟早上的零食預算題一樣是三件事：

- 決策變數：每兩個點之間要不要走這一段，也就是每件東西撿不撿
- 限制式：要走成一條從自己出發、回到撤離點的路線（`AddCircuit`），背包裝得下
- 目標：撿到的分數，減掉走路的秒數乘上 `per_second`（每秒值幾分）

執行這一格，再把「4. 本地測試」裡的 `my_make_policy` 換成 `cpsat_make_policy` 比比看。
可以試著改 `per_second`、`max_tasks`，或在目標裡加上離殭屍越近扣越多分。

### 這些數字怎麼放進 OR-Tools

| 模型裡的東西 | 從哪裡拿 |
|---|---|
| 每件東西值多少分 | 零件：`t["reward"] * 80 / (80 + 我交過的零件數)`；民眾、油桶：`t["reward"]`；趕得上期限的話再加 `t["bonus"]` |
| 事件值多少分 | `inc["reward"] * 2 / (一起做的人數)`，一起做的人數 = `st.helpers(inc["id"]) + inc["workers"] + 1` |
| 每件東西多重 | `t["weight"]` |
| 背包還能裝多少 | `st.rules["capacity"] - st.you["load"]` |
| 走一段要幾秒 | `g.dist(a, b) / st.you["speed"]` |
| 事件要做幾秒 | `(inc["need"] - inc["progress"]) / 一起做的人數` |
| 每秒值幾分（`per_second`） | 一場打得好大約 3000 分 ÷ 600 秒，也就是每秒 3 到 6 分 |

`我交過的零件數` 可以這樣拿：`next(p for p in st.players if p["id"] == st.you["id"])["parts"]`。

`per_second` 是「多走一秒要扣幾分」：設太大，只會撿身邊的小東西；設太小，會跑很遠去撿一大堆。可以用「4. 本地測試」試幾個數字比比看。

送出計畫後要等 15 秒才能再送，所以一次排的路線最好能走 15 秒以上。求解時間設 0.3 秒左右就夠了。

In [ ]:
#@title 6. OR-Tools 範例：CP-SAT
from ortools.sat.python import cp_model

def plan_trip(st, g, tasks, capacity, speed, per_second=6.0, max_tasks=8):
    ex = st.extraction

    def secs(a, b):
        return g.dist(a, b) / speed

    def single_rate(t):
        return t["reward"] / (secs(st.here, t["origin"]) + secs(t["origin"], ex) + 2)
    cands = sorted(tasks, key=single_rate, reverse=True)[:max_tasks]
    if not cands:
        return [], 0

    nodes = [st.here] + [t["origin"] for t in cands] + [ex]
    n = len(cands)
    END = n + 1

    m = cp_model.CpModel()
    arcs, arc_vars = [], {}
    for i in range(END + 1):
        for j in range(END + 1):
            if i == j or j == 0 or i == END:
                continue
            v = m.NewBoolVar(f"go_{i}_{j}")
            arc_vars[i, j] = v
            arcs.append((i, j, v))
    arcs.append((END, 0, m.NewConstant(1)))
    take = []
    for k in range(1, n + 1):
        skip = m.NewBoolVar(f"skip_{k}")
        arcs.append((k, k, skip))
        take.append(skip.Not())
    m.AddCircuit(arcs)

    m.Add(sum(cands[k]["weight"] * take[k] for k in range(n)) <= capacity)

    value = sum(int(cands[k]["reward"] * 10) * take[k] for k in range(n))
    walk = sum(int((secs(nodes[i], nodes[j]) + (1 if j <= n else 0)) * 10) * v for (i, j), v in arc_vars.items())
    m.Maximize(value - int(per_second) * walk)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 0.3
    if solver.Solve(m) not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return [], 0

    order, at, total = [], 0, 0.0
    while at != END:
        nxt = next(j for (i, j), v in arc_vars.items() if i == at and solver.Value(v))
        total += secs(nodes[at], nodes[nxt])
        if nxt != END:
            order.append(cands[nxt - 1])
        at = nxt
    return order, total

def cpsat_make_policy(per_second=6.0):
    def policy(st):
        me = st.you
        g = CityGraph(st, zombie_penalty=3, zombie_lookahead=10)
        ex, depart = st.extraction, st.escape["depart_at"]

        if depart > 0 and depart - st.time < g.dist(st.here, ex) / me["speed"] + 15:
            if st.here == ex and st.is_idle:
                drops = [dropoff(t["id"]) for t in st.my_cargo() if t["dest"] == ex]
                return drops if drops and st.time >= me["next_plan_at"] else None
            going = me["plan"] and me["plan"][-1]["node"] == ex
            return None if going else [goto(ex)]
        if not st.is_idle:
            return None
        if me["hp"] < 40 or me["infection"] > 60:
            if me["medkits"] > 0:
                return [use_medkit()]
            return [goto(g.nearest(st.here, st.nodes_of("hospital"))), heal()]
        e = st.escape
        useless = [t for t in st.my_cargo() if (t["type"] == "part" and e["ready_at"] >= 0)
                   or (t["type"] == "fuel" and e["fuel"] >= e["fuel_required"])]
        if useless:
            return [drop(t["id"]) for t in useless]
        if st.time < me["next_plan_at"]:
            return None

        for q in st.players:
            if q["state"] == "downed" and q["id"] != me["id"]:
                eta = g.dist(st.here, q["node"]) / me["speed"]
                others = [g.dist(o["node"], q["node"]) for o in st.players
                          if o["state"] == "active" and o["id"] not in (me["id"], q["id"])]
                if eta + 4 < q["downed_timer"] and g.dist(st.here, q["node"]) <= min(others, default=1e9):
                    return [goto(q["node"]), revive(q["id"])]

        cargo = st.my_cargo()
        if cargo:
            return [s for t in cargo for s in (goto(t["dest"]), dropoff(t["id"]))]

        for inc in sorted(st.active_incidents(), key=lambda i: g.dist(st.here, i["node"])):
            eta = g.dist(st.here, inc["node"]) / me["speed"]
            if (eta < 25 or inc["type"] == "generator") and eta + 3 < inc["deadline"] - st.time and st.helpers(inc["id"]) < inc["min_people"]:
                return [goto(inc["node"]), fix(inc["id"], 20)]

        claimed = st.claimed_tasks()
        tasks = [t for t in st.open_tasks() if t["id"] not in claimed and t["dest"] == ex]
        if st.escape.get("ready_at", -1) >= 0:
            tasks = [t for t in tasks if t["type"] == "fuel"] if st.escape["fuel"] < st.escape["fuel_required"] else []
            if not tasks:
                return None if st.here == ex else [goto(ex)]
        if not tasks:
            return None

        capacity = st.raw["rules"]["capacity"] - me.get("load", 0)
        order, _ = plan_trip(st, g, tasks, capacity, max(0.5, me["speed"]), per_second)
        if not order:
            order = [min(tasks, key=lambda t: g.dist(st.here, t["origin"]))]
        plan = []
        for t in order:
            plan += [goto(t["origin"]), pickup(t["id"])]
        plan += [goto(ex)] + [dropoff(t["id"]) for t in order]
        return plan

    return policy